# Project Task: Compositional Image Retrieval
This Notebook explores the task of compositional image retrieval, where the goal is to retrieve images based on a combination of visual and textual inputs. 

We present and evaluate different approaches to this task, comparing their performance with a baseline method.

## Environment Setup

This notebook is designed for Google Colab.

Before running, ensure:
- Google Drive is mounted.
- The dataset zip exists at `/content/drive/MyDrive/datasets/celeba.zip`.
- You run cells top-to-bottom at least once to initialize all variables.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Create the directory if it doesn't exist
!mkdir -p /content/datasets

In [ ]:
# This should take 1-2 minutes
# It unzips the dataset in the runtime's local SSD, so when
# you disconnect, it gets deleted
!unzip -q /content/drive/MyDrive/datasets/celeba.zip -d /content/datasets/

### Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import os
import math
import json
from typing import Callable
import matplotlib.pyplot as plt
import numpy as np

from transformers import CLIPModel, CLIPProcessor
from torchvision.datasets import CelebA

#### Device configuration

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
MODEL_NAME = "openai/clip-vit-base-patch32"

### Encoding utilities
Common CLIP encoding utilities for processing images and text, shared across the notebook. 
These wrap the CLIP model for encoding both images and text into a common embedding space, with a single source of truth for encoding logic.

In [ ]:
_model = None
_processor = None


def get_CLIP_model():
    global _model, _processor
    if _model is None:
        print("Loading CLIP model...")
        _model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
        _model.eval()
    if _processor is None:
        _processor = CLIPProcessor.from_pretrained(MODEL_NAME)
    return _model, _processor


def _as_feature_tensor(out) -> torch.Tensor:
    """
    Normalize CLIPModel.get_text_features / get_image_features outputs into a Tensor.
    Different transformers versions return either a plain Tensor or a
    BaseModelOutputWithPooling-style object exposing .text_embeds / .image_embeds /
    .pooler_output.
    """
    if isinstance(out, torch.Tensor):
        return out
    for attr in ("text_embeds", "image_embeds", "pooler_output"):
        v = getattr(out, attr, None)
        if v is not None:
            return v
    if isinstance(out, tuple) and len(out) > 0:
        return out[0]
    raise TypeError(f"Unexpected feature output type: {type(out)}")


@torch.no_grad()
def encode_texts(prompts: list[str], device) -> torch.Tensor:
    """Encode a batch of prompts in one call. Returns (P, D), L2-normalized per row, on `device`."""
    model, processor = get_CLIP_model()
    inputs = processor(text=prompts, return_tensors="pt", padding=True, truncation=True).to(device)
    embs = _as_feature_tensor(model.get_text_features(**inputs))
    return F.normalize(embs, p=2, dim=-1)


@torch.no_grad()
def encode_text(prompt: str, device) -> torch.Tensor:
    """Tokenize, encode, and L2-normalize a single text prompt. Returns shape (D,) on `device`."""
    return encode_texts([prompt], device).view(-1)


@torch.no_grad()
def encode_images(images, device, batch_size: int = 128, to_cpu: bool = True, log_every: int = 1000) -> torch.Tensor:
    """Encode an iterable of PIL images. Returns (N, D), L2-normalized per row.
    Streams in `batch_size` chunks so callers don't have to hand-roll batching.
    Each batch is moved to CPU before concatenation by default to keep GPU memory
    bounded when encoding large datasets. Pass `to_cpu=False` to keep the result
    on `device` (useful for single-image / small-batch calls).
    """
    model, processor = get_CLIP_model()
    images = list(images)
    n_total = len(images)

    feats: list[torch.Tensor] = []
    buf: list = []
    n_encoded = 0

    def _flush():
        nonlocal n_encoded
        inputs = processor(images=buf, return_tensors="pt").to(device)
        e = _as_feature_tensor(model.get_image_features(**inputs))
        e = F.normalize(e, p=2, dim=-1)
        feats.append(e.cpu() if to_cpu else e)
        n_encoded += len(buf)
        buf.clear()

    for img in images:
        buf.append(img)
        if len(buf) >= batch_size:
            _flush()
            if n_encoded % log_every < batch_size:
                pct = 100 * n_encoded / n_total
                print(f"Encoded {n_encoded:>{len(str(n_total))}}/{n_total} images ({pct:.1f}%)")

    if buf:
        _flush()

    print(f"Encoded {n_encoded}/{n_total} images (100.0%) — done.")
    return torch.cat(feats, dim=0) if feats else torch.empty(0)


def _collate_keep_pil(batch):
    """Keep PIL images as a Python list; only stack the label tensors.
    The default collate_fn cannot stack PIL.Image objects, and the encode
    loop wants a list of PIL images to feed to the CLIP processor."""
    imgs = [item[0] for item in batch]
    lbls = torch.stack([item[1] for item in batch], dim=0)
    return imgs, lbls


@torch.no_grad()
def get_encoded_dataset(
    dataset,
    device,
    cache_path: str,
    batch_size: int = 128,
    num_workers: int = 4,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Encode all images in `dataset` and return (features, labels).
      - features: (N, D) on `device`, L2-normalized per row.
      - labels:   (N, ...) on CPU, as produced by the dataset.

    If a cached file exists at `cache_path`, load and return it. Otherwise
    compute features and labels in a single DataLoader pass, cache as a dict
    {"features", "labels"}, and return. Legacy caches that hold a raw features
    tensor (no labels) are still readable: labels are re-stacked from `dataset`
    without re-encoding images, and the cache file is left untouched.
    """
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)

    if os.path.exists(cache_path):
        print(f"Loading cached features from {cache_path}.")
        blob = torch.load(cache_path, map_location="cpu")
        if isinstance(blob, dict):
            features = blob["features"].to(device)
            labels   = blob["labels"]
        else:
            # Legacy cache: features only — re-stack labels without re-encoding.
            features = blob.to(device)
            labels   = torch.stack([lbl for _img, lbl in dataset], dim=0)
        print(f"Loaded from cache. features: {tuple(features.shape)}, labels: {tuple(labels.shape)}")
        return features, labels

    print("Cache not found. Encoding dataset...")
    model, processor = get_CLIP_model()

    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True,
        shuffle=False,
        collate_fn=_collate_keep_pil,
    )

    feats_list: list[torch.Tensor] = []
    lbls_list:  list[torch.Tensor] = []
    n_encoded = 0
    n_total   = len(dataset)
    pad       = len(str(n_total))

    for imgs_batch, lbls_batch in loader:
        inputs = processor(images=list(imgs_batch), return_tensors="pt").to(device)
        e = _as_feature_tensor(model.get_image_features(**inputs))
        e = F.normalize(e, p=2, dim=-1)
        feats_list.append(e.cpu())
        lbls_list.append(lbls_batch)
        n_encoded += len(imgs_batch)
        print(f"Encoded {n_encoded:>{pad}}/{n_total} images ({100 * n_encoded / n_total:.1f}%)")

    features = torch.cat(feats_list, dim=0).to(device)
    labels   = torch.cat(lbls_list,  dim=0)

    torch.save({"features": features.cpu(), "labels": labels}, cache_path)
    print(f"Saved to {cache_path}. features: {tuple(features.shape)}, labels: {tuple(labels.shape)}")
    return features, labels

def _clip_encode_token_emb(
    clip_model: "CLIPModel",
    token_emb: torch.Tensor,
    input_ids: torch.Tensor,
) -> torch.Tensor:
    """Run the frozen CLIP text transformer on a pre-built token-embedding matrix.

    Both CoOp and Pic2Word overwrite specific token positions before calling this
    function; all other transformer machinery (position embeddings, causal mask,
    encoder, LN, EOS pooling, projection, L2-normalise) is shared here.

    Args:
        clip_model: the frozen HuggingFace CLIPModel.
        token_emb:  (B, L, D) token embedding tensor with any overwrites applied.
        input_ids:  (B, L) token-id tensor used only to locate the EOS position.
    Returns:
        (B, D) L2-normalised text features.
    """
    text_model = clip_model.text_model
    device_    = token_emb.device
    bsz, seq_len = input_ids.shape

    position_ids = torch.arange(seq_len, device=device_).unsqueeze(0)
    hidden_states = token_emb + text_model.embeddings.position_embedding(position_ids)

    causal_mask = torch.full((seq_len, seq_len), float("-inf"), device=device_)
    causal_mask = torch.triu(causal_mask, diagonal=1)
    causal_mask = causal_mask.unsqueeze(0).unsqueeze(0).expand(bsz, 1, seq_len, seq_len)

    encoder_outputs = text_model.encoder(
        inputs_embeds=hidden_states,
        attention_mask=None,
        causal_attention_mask=causal_mask,
        output_attentions=False,
        output_hidden_states=False,
        return_dict=True,
    )
    last_hidden = text_model.final_layer_norm(encoder_outputs.last_hidden_state)

    eos_pos = input_ids.argmax(dim=-1)   # CLIP convention: pool at the EOS token
    pooled  = last_hidden[torch.arange(bsz, device=device_), eos_pos]
    text_features = clip_model.text_projection(pooled)
    return text_features / text_features.norm(dim=-1, keepdim=True)


### Plotting utilities
Helper functions for visualizing retrieved images and their associated prompts, used across the notebook for consistent presentation of results.

In [ ]:
def plot_images(celeba_dataset: object, indices: list[int], n_cols: int, n_rows: int, figsize: tuple[int, int]=(20, 10)):
    """Utility function to plot a grid of images given their indices in the CelebA dataset.
    Args:
        celeba_dataset: The CelebA dataset object.
        indices: A list of indices corresponding to the images to be plotted.
        n_cols: Number of columns in the grid.
        n_rows: Number of rows in the grid.
        figsize: Size of the figure (width, height).
    """
    if len(indices) > n_cols * n_rows:
        raise ValueError("Number of indices exceeds the grid capacity")
    
    _, axes = plt.subplots(n_rows, n_cols, figsize=figsize)

    for counter, img_idx in enumerate(indices):
        img, _ = celeba_dataset[img_idx]
        if n_rows == 1:
            ax = axes[counter % n_cols]
        else:
            ax = axes[counter // n_cols, counter % n_cols]
        ax.imshow(img)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

### Results visualization
Functions for visualizing the results of image retrieval, including displaying retrieved images alongside their prompts and similarity scores.

In [14]:
def plot_metrics_across_k(average_results_per_query: list[dict], title: str = "Retrieval Metrics across K"):
    """
    Plot Recall@K and Precision@K as a grouped bar chart: one bar per query, grouped by K, with 95% confidence intervals.
    Args:
        average_results_per_query: Output from compute_query_average_results() for each query — list of per-query average dicts.
        title: Title for the overall figure.
    """
    k_values = [1, 5, 10]
    n_queries = len(average_results_per_query)
    x = np.arange(n_queries)
    width = 0.25
    offsets = [-width, 0.0, width]
    colors = [plt.cm.tab10(i) for i in range(len(k_values))]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(title)

    for k, offset, color in zip(k_values, offsets, colors):
        recall_means    = [q[f"Recall@{k}"]       for q in average_results_per_query]
        recall_cis      = [q[f"Recall@{k}_CI"]    for q in average_results_per_query]
        precision_means = [q[f"Precision@{k}"]    for q in average_results_per_query]
        precision_cis   = [q[f"Precision@{k}_CI"] for q in average_results_per_query]

        ax1.bar(x + offset, recall_means,    width, yerr=recall_cis,    capsize=4, ecolor="black", color=color, label=f"K={k}")
        ax2.bar(x + offset, precision_means, width, yerr=precision_cis, capsize=4, ecolor="black", color=color, label=f"K={k}")

    for ax, metric in [(ax1, "Recall"), (ax2, "Precision")]:
        ax.set_xlabel("Query")
        ax.set_ylabel(f"{metric}@K")
        ax.set_title(f"{metric}@K per query")
        ax.set_xticks(x)
        ax.set_xticklabels([f"Q{i+1}" for i in range(n_queries)])
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3, axis="y")
        ax.legend(title="K")

    plt.tight_layout()
    plt.show()


def plot_methods_comparison(method_results: dict[str, list[dict]], title: str = "Method Comparison across queries"):
    """
    Per-query line comparison of N retrieval methods. Layout is a 2 x 3 grid:
    rows are Recall (top) and Precision (bottom); columns are K in {1, 5, 10}.
    In each subplot, one segmented line per method connects the metric values
    over the query axis, so per-query differences between methods are visible.
    Args:
        method_results: Dict mapping method name -> average_results_per_query (same shape consumed by plot_metrics_across_k).
        title: Title for the overall figure.
    """
    k_values = [1, 5, 10]
    method_names = list(method_results.keys())
    n_methods = len(method_names)
    if n_methods == 0:
        raise ValueError("method_results must contain at least one method.")

    n_queries = len(next(iter(method_results.values())))
    x = np.arange(n_queries)
    colors = [plt.cm.tab10(i % 10) for i in range(n_methods)]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
    fig.suptitle(title)

    for row_idx, metric in enumerate(["Recall", "Precision"]):
        for col_idx, k in enumerate(k_values):
            ax = axes[row_idx, col_idx]
            for method, color in zip(method_names, colors):
                ys = [q[f"{metric}@{k}"] for q in method_results[method]]
                ax.plot(x, ys, marker="o", color=color, label=method)
            ax.set_title(f"{metric}@{k}")
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3)
            ax.set_xticks(x)
            ax.set_xticklabels([f"Q{i+1}" for i in range(n_queries)], rotation=45, ha="right")

    for ax in axes[:, 0]:
        ax.set_ylabel("Score")
    for ax in axes[-1, :]:
        ax.set_xlabel("Query")

    axes[0, 0].legend(title="Method", loc="best")

    plt.tight_layout()
    plt.show()

## Data loading and Exploration
In this section, we load the dataset and perform an initial exploration to understand its structure and main characteristics.

The dataset used is `CelebA`, which contains 19,962 samples. Each sample consists of:
- an image of size `178 × 218`, representing a face of a celebrity;
- a set of 40 attributes that describe visual features of the person in the image.

We will briefly inspect the dataset by visualizing some samples and examining the associated attributes, in order to get a better understanding of the data before proceeding with further analysis.

In [ ]:
# Do *not* put `celeba` in the path.
# The dataset class will do that automatically!
data_root = Path("/content/datasets")
celeba = CelebA(root=data_root, split="test", download=False)


def get_attribute_names(dataset) -> list[str]:
    """Return the dataset's attribute names, dropping torchvision's spurious empty entry.

    torchvision's CelebA exposes `attr_names` with 41 entries (one is an empty string),
    while the label tensor has 40 columns. Dropping empties keeps the names aligned with
    the label columns and with the learned per-attribute embedding rows.
    """
    return [name for name in dataset.attr_names if name]

# This should be 19.962
print("Number of samples:", len(celeba))

# show element size
sample_img, sample_attrs = celeba[0]
print(f"Sample image size: {sample_img.size}")
print(f"Number of attributes: {len(sample_attrs)}")

### Sample visualization
First, we can visualize a random selection of images from the dataset to get a sense of the variety and quality of the images. We will display 50 random images in a grid format.

In [ ]:
# Get 50 random images and visualize them
indices = np.random.choice(len(celeba), size=50, replace=False)
plot_images(celeba, indices=indices, n_cols=10, n_rows=5)

### Attribute annotation
Now that we know how to load our dataset and we have visualized some samples, let's move to understanding how attributes are annotated in the dataset. Each image in the dataset is annotated with a set of 40 binary attributes, from the following list. 

Here, we also report how frequently each attribute appears in the dataset, which is important to understand the distribution of attributes and to design a retrieval system that can handle rare attributes effectively.


In [ ]:
all_labels = np.array([labels for _, labels in celeba])

attr_counts = all_labels.sum(axis=0)
attr_freq = all_labels.mean(axis=0)

print(f"{'Attribute':<20} {'Count':>10} {'Frequency':>10}")
print("-" * 45)

for attr, count, freq in zip(celeba.attr_names, attr_counts, attr_freq):
    print(f"{attr:<20} {count:>10} {freq:>10.3f}")

Let's define few other utilities functions that will facilitate the handling of attributes later on.
We will create a:
- `inx2attribute`: mapping from indices to attributes (e.g., index 0 corresponds to "5_o_Clock_Shadow", index 1 corresponds to "Arched_Eyebrows", etc.)

- `attribute2index`: mapping from attributes to indices (e.g., "5_o_Clock_Shadow" corresponds to index 0, "Arched_Eyebrows" corresponds to index 1, etc.)

- `retrieve_by_attributes(parameters)`: function that retrieves images based on specified attributes. This function will be crucial for our image retrieval system, allowing us to query the dataset for images that match certain attribute criteria.

- `plot_image_with_attributes`: function that displays an image along with its active attributes.

Query format for `retrieve_by_attributes`:
- Use `"+"` when the attribute must be present.
- Use `"-"` when the attribute must be absent.

Example: `{"Bald": "+", "Eyeglasses": "-"}`.

In [ ]:
# Assign a unique index to each attribute, and get the inverse mapping.
# Drop the spurious trailing empty name that torchvision's CelebA exposes
# (attr_names has 41 entries, one empty) so indices stay 0..39 and aligned
# with the 40-column label tensor and the learned CoOp embedding rows.
attr_names_clean = get_attribute_names(celeba)
idx2attribute = {idx: name for idx, name in enumerate(attr_names_clean)}
attribute2idx = {name: idx for idx, name in enumerate(attr_names_clean)}

def retrieve_by_attributes(parameters:dict):
    """
    Helper function that retrieve all the images that satisfy the conditions specified 
    in `parameters`.
    Params:
        - parameters: a dictionary where keys are attribute names and values are either "+" (must have the attribute) or "-" (must not have the attribute).  
    Returns:
        - A list of indices of images that satisfy the specified conditions.
    """
    # Start with all indices
    valid_indices = set(range(len(celeba)))
    
    # For each attribute condition, filter the indices
    for attr_name, value in parameters.items():
        attr_idx = attribute2idx[attr_name]
        if value == "+":
            for idx in valid_indices.copy():
                if celeba[idx][1][attr_idx] == 0:
                    valid_indices.remove(idx)
        elif value == "-":
            # Must not have the attribute
            for idx in valid_indices.copy():
                if celeba[idx][1][attr_idx] == 1:
                    valid_indices.remove(idx)
        else:
            raise ValueError(f"Invalid value for attribute condition: {value}. Use '+' or '-'.")
    
    return list(valid_indices)

def plot_image_with_attributes(idx: int, figsize: tuple[int, int]=(10, 5)):
    """Helper function to plot a single image along with its active attributes as text on the right side of the image."""
    img, labels = celeba[idx]
    active_attrs = [idx2attribute[idx] for idx, value in enumerate(labels) if value == 1]

    fig, (ax_img, ax_text) = plt.subplots(1, 2, figsize=figsize)
    # Left: image
    ax_img.imshow(img)
    ax_img.axis('off')

    # Right: centered text
    ax_text.axis('off')
    text = "\n".join(active_attrs)

    ax_text.text(
        0.5, 0.5, text,
        fontsize=10,
        ha='center',   # horizontal alignment
        va='center'    # vertical alignment
    )

    plt.tight_layout()
    plt.show()

Now that we have the mapping, we can easily get the attributes of any image in the dataset. For example, let's get the attributes of a given image index.

In [ ]:
IMAGE_INDEX = 99
plot_image_with_attributes(IMAGE_INDEX)

Now that we have everything in place, let's try to analyze some possible queries.


In [ ]:
query_1 = {"Bald": "+",
           "Smiling": "+",
           "Eyeglasses": "-",
           }
retrieved_images = retrieve_by_attributes(query_1)
print(f"Number of retrieved images: {len(retrieved_images)}")

# Plot up to 10 random retrieved images (without replacement).
n_samples = min(10, len(retrieved_images))
if n_samples == 0:
    print("No images match this query.")
else:
    sampled_indices = np.random.choice(retrieved_images, size=n_samples, replace=False)
    n_cols = 5
    n_rows = int(np.ceil(n_samples / n_cols))
    plot_images(celeba, indices=sampled_indices, n_cols=n_cols, n_rows=n_rows)

# Offline Feature Extraction
In this section we extract features from our dataset using a Vision Language Model (VLM).

This representation is computed once and then kept frozen offline. The goal is not to improve the VLM encoder itself, but to study and improve retrieval behavior on top of fixed CLIP embeddings.

In [ ]:
EVALUATION_CACHE_DIR = "/content/drive/MyDrive/datasets/clip_cache"
EVALUATION_CACHE_PATH = os.path.join(EVALUATION_CACHE_DIR, "embeddings.pt")

### Extracting features using a VLM
In this section we will be using the CLIP model to extract features from our dataset. We will be using the ViT-B/32 model, which is a smaller version of the original CLIP model. 

Since these embeddings are static, we can compute them offline and keep them frozen. This means that we don't have to worry about the computational cost of computing the embeddings during training, and we can also use a larger batch size for training our retriever model.

The result of this process will be a list of embeddings, where each embedding is of size 512 and corresponds to an image in our dataset. 

In [ ]:
# Get the encoded dataset, using cached features if available
embeddings, embedding_labels = get_encoded_dataset(celeba, device, EVALUATION_CACHE_PATH, batch_size=128)


## Sanity check
Now that embeddings for the image dataset are available, let's run a quick sanity check to verify retrieval quality.

We will pick a source image, compare its CLIP embedding against all dataset embeddings, and inspect the nearest matches.

In [ ]:
source_idx = 10006
img, _ = celeba[source_idx]

plt.figure(figsize=(4, 4))
plt.axis('off')
plt.imshow(img)

Now that we have our source image and its CLIP encoding, let's find the nearest embeddings in the dataset.

We normalize embeddings to unit vectors at extraction time, so similarity between any two embeddings reduces to a plain dot product.
For unit vectors, `dot(a, b) == cosine_similarity(a, b)`, so the metric is unchanged — but retrieval becomes a single matrix multiply instead of a per-pair loop.
We exclude the source image itself from the top results.

In [ ]:
if "embeddings" not in globals():
    raise RuntimeError(
        "Embeddings not found. Run the offline feature extraction cell above first."
    )

source_embedding = embeddings[source_idx]

# Dot product == cosine similarity for unit-norm embeddings (single matrix-vector multiply)
# 19962 x 512 @ 512 -> 19962, all on GPU.
similarities = embeddings @ source_embedding

# Get the 6 highest-similarity matches and drop the source itself.
top_vals, top_idx = torch.topk(similarities, k=6)
nearest_indices = top_idx[1:].tolist()         # Python ints needed to index `celeba[...]`
nearest_similarities = top_vals[1:].tolist()   # Python floats needed for plot titles

print("Nearest indices:", nearest_indices)
print("Nearest cosine similarities:", nearest_similarities)


We have found the nearest embeddings! Let's visualize the nearest images to our source image and see if they are indeed similar.

In [ ]:
fig, axes = plt.subplots(ncols=5, figsize=(25, 5))

for i, image_idx in enumerate(nearest_indices):
    img, labels = celeba[image_idx]
    ax = axes[i]
    
    ax.set_title(f"Cosine sim: {nearest_similarities[i]:.4f}")
    
    ax.imshow(img)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Embedding Analysis: Class × Image Cosine Heatmap

Before introducing any retrieval method, it is useful to inspect how well CLIP separates the 40 CelebA attributes on real images.

For every attribute we:
1. Encode the naturalized prompt through the CLIP text encoder.
2. Pick **one "pure" image** from the dataset: an image where this attribute is active and the count of other co-active attributes is minimal. This makes the diagonal of the resulting heatmap interpretable as a true match while reducing visual confusion from CelebA's many overlapping labels.
3. Build the 40×40 matrix of cosine similarities between the 40 text embeddings and the 40 selected image embeddings.

If CLIP is well-aligned with these attribute concepts, the heatmap diagonal should dominate each row. Strong off-diagonal cells highlight attribute pairs CLIP confuses on this dataset (e.g. `Wavy_Hair` vs `Straight_Hair`, `Heavy_Makeup` vs `Wearing_Lipstick`).

In [ ]:
def _select_pure_image_idxs(all_labels: np.ndarray, rng: np.random.Generator) -> list[int]:
    """For each attribute, pick one image where it is positive and the count of other positive attributes is minimal."""
    selected = []
    for attr_idx in range(all_labels.shape[1]):
        candidates = np.where(all_labels[:, attr_idx] == 1)[0]
        if len(candidates) == 0:
            selected.append(int(rng.integers(0, all_labels.shape[0])))
            continue
        other_counts = all_labels[candidates].sum(axis=1) - 1
        purest = candidates[other_counts == other_counts.min()]
        selected.append(int(rng.choice(purest)))
    return selected


def _plot_cosine_heatmap(cos_mat: np.ndarray, attr_names: list[str]) -> None:
    """Plot a cosine similarity heatmap of text prompts vs sampled images."""
    n = len(attr_names)
    fig, ax = plt.subplots(figsize=(14, 14))
    im = ax.imshow(cos_mat, cmap="viridis", aspect="equal")
    ticks = np.arange(n)
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels(attr_names, rotation=90, fontsize=8)
    ax.set_yticklabels(attr_names, fontsize=8)
    ax.set_xlabel("Sampled image (chosen as 'pure' positive for this attribute)")
    ax.set_ylabel("Text prompt: 'A picture of a person with {attr}'")
    ax.set_title(f"CLIP cosine similarity: {n} attribute prompts × {n} sampled images")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="cosine similarity")
    plt.tight_layout()
    plt.show()


def _print_cosine_diagnostics(cos_mat: np.ndarray) -> None:
    """Print diagonal vs off-diagonal cosine similarity statistics."""
    n = cos_mat.shape[0]
    diag = np.diag(cos_mat)
    off_diag_mean = (cos_mat.sum() - diag.sum()) / (cos_mat.size - n)
    diag_argmax_rate = float((cos_mat.argmax(axis=1) == np.arange(n)).mean())
    print(f"Mean diagonal cosine:      {diag.mean():.4f}")
    print(f"Mean off-diagonal cosine:  {off_diag_mean:.4f}")
    print(f"Diagonal-argmax rate:      {diag_argmax_rate:.2%} (attributes where the matching image is the row's argmax)")


attr_names = get_attribute_names(celeba)   # sanitized: drops torchvision's spurious empty entry
prompts = [f"{name.replace('_', ' ').lower()}" for name in attr_names]
text_embs = encode_texts(prompts, device)  # (n_attrs, 512)


rng = np.random.default_rng(seed=0)
selected_idxs = _select_pure_image_idxs(all_labels, rng)
selected_img_embs = embeddings[selected_idxs].to(text_embs.device)  # (n_attrs, 512)


cos_mat = (text_embs @ selected_img_embs.T).detach().cpu().numpy()  # (n_attrs, n_attrs)
_print_cosine_diagnostics(cos_mat)

#### Cosine heatmap: visualisation

In [ ]:
_plot_cosine_heatmap(cos_mat, attr_names)

## Metrics
In this section we will define the metrics that we will use to evaluate our retrieval system.

In particular, we will be using the following metrics:
- **Recall@K**: This metric measures if **at least one** valid truth image is present in the top K retrieved images. It is defined as:
$$Recall@K = \begin{cases} 1 & \text{if } |\mathcal{R}_K \cap \mathcal{G}| > 0 \\ 0 & \text{otherwise} \end{cases}$$
- **Precision@K**: This metric measures the proportion of relevant images in the top K retrieved images. It is defined as:
$$Precision@K = \frac{| \mathcal{R}_K \cap \mathcal{G} |}{K}$$

Where:
- $\mathcal{R}_K$ is the set of top K retrieved images for the given query.
- $\mathcal{G}$ is the set of ground truth relevant images for the given query.



In [ ]:
def evaluate_retrieval(
    retrieved_indices: list[int],
    ground_truth_indices: list[int],
    k: int
) -> dict:
    """
    Evaluate the retrieval performance for a single source image.
    Args:
        - retrieved_indices: list of image IDs predicted by the model, ordered by similarity (descending).
        - ground_truth_indices: list of valid target IDs from the benchmark JSON.
        - k: the cutoff for top-K evaluation (e.g., 1, 5, 10).
    Return:
        - A dictionary containing Recall@K and Precision@K metrics.

    """
    # Get the top K retrieved indices
    #NOTE: the retrieved_indices must be ordered by similarity in descending order
    top_k_retrieved = retrieved_indices[:k]

    # Calculate the intersection between predictions and ground truth
    hits = set(top_k_retrieved).intersection(set(ground_truth_indices))
    num_hits = len(hits)

    # Metrics calculations
    # Recall@K (Hit Rate): 1 if at least one match is found, 0 otherwise
    recall_at_k = 1 if num_hits > 0 else 0

    # Precision@K: Fraction of top K predictions that are correct
    precision_at_k = num_hits / k

    return {
        f"Recall@{k}": recall_at_k,
        f"Precision@{k}": precision_at_k
    }


def mean_recall_at_10(evaluation_results: list[dict]) -> float:
    """Mean Recall@10 over every (query, source image) pair — the scalar we use
    to compare hyperparameter settings across the training-free methods."""
    vals = [
        metrics[10]["Recall@10"]
        for query_results in evaluation_results
        for metrics in query_results.values()
    ]
    return float(np.mean(vals))


#### Example usage

In [ ]:
# --- Example Usage ---
# Suppose the model returns these indices from most to least similar:
predictions = [1, 2, 3, 4, 5]
# And we load this from our JSON for this specific source:
ground_truth = [3, 2, 1]

# Evaluate at K=1 and K=5
print("Results @ 1:", evaluate_retrieval(predictions, ground_truth, k=1))
print("Results @ 5:", evaluate_retrieval(predictions, ground_truth, k=5))

## Evaluation Protocol
To assess the performance of our retrieval system, we utilize a standardized benchmark of queries stored in a JSON file. Each entry in the dataset follows this structure:

* **`query`**: A string representing the textual modification (e.g., `"+glasses, -smiling"`).
* **`ground_truth`**: A dictionary where:
    * **Keys** are the indices of the **source images** used as the starting point.
    * **Values** are lists of indices for images considered valid retrievals for that specific source.

### Example Structure
```json
{
    "query": "+glasses, -smiling",
    "ground_truth": {
        "0": [1, 2, 3],
        "4": [5, 6, 7]
    }
}
```
In this example, image 0 serves as a source image (e.g., a smiling person without glasses). The system is expected to retrieve images 1, 2, or 3, which represent the "target" state (a non-smiling person with glasses), which should be visually similar to the source image but with the specified modifications applied.

In [ ]:
# Open the JSON file containing the benchmark annotations
annotations_path = Path("/content/drive/MyDrive/datasets/celeba_evaluation.json")
with open(annotations_path, "r") as f:
    annotations = json.load(f)

# Print the number of annotations loaded
print(f"Loaded {len(annotations)} queries!")

We can define some utility functions to facilitate the evaluation process

In [ ]:
# Display a sample annotation to understand the structure of the data
print("Sample annotation shape", annotations[0].keys())

# Extract and print first text query
print("Text-Query example:", annotations[0].get("query", ""))

# Extract and print the source image ID for the first annotation
print("Source-Image example:", list(annotations[0].get("ground_truth", {}).keys())[:5],"...")

# Extract and print the list of ground truth indices for the first annotation
print("List of ground truth indices for the first annotation:", annotations[0].get("ground_truth", {}).get("13", [])[:5], "...")

def get_text_query(annotation: dict) -> str:
    """
    Helper function to extract the text query from a benchmark annotation.
    Args:
        - annotation: A dictionary containing the benchmark annotation for a single query.
    Returns:
        - A string representing the text query (e.g., "+glasses, -smile").
    """
    return annotation.get("query", "")

def get_source_image_idxs(annotation: dict) -> list[int]:
    """
    Helper function to extract the source image IDs from a benchmark annotation.
    Args:
        - annotation: A dictionary containing the benchmark annotation for a single query.
    Returns:
        - A list of integers representing the source image IDs.
    """    
    # The key of the "ground_truth" must be converted to int since JSON keys are always strings
    return [int(key) for key in annotation.get("ground_truth", {}).keys()]

def get_ground_truth_indices(annotation: dict, source_image_idx: int) -> list[int]:
    """
    Helper function to extract the list of valid target IDs from a benchmark annotation.
    Args:
        - annotation: A dictionary containing the benchmark annotation for a single query.
        - source_image_idx: The index of the source image for which to retrieve ground truth indices.
    Returns:
        - A list of valid target IDs (integers) that are considered correct matches for the given query.
    """
    return annotation.get("ground_truth", {}).get(str(source_image_idx), [])


#### Sanity-check annotation helpers

In [ ]:
# Let's test these utility functions on the first annotation in the dataset
annotation = annotations[1]

text_query = get_text_query(annotation)
print("Text query:", text_query )

source_image_idx = get_source_image_idxs(annotation)[0]
print("Source image index:", source_image_idx)
plot_image_with_attributes(source_image_idx, figsize=(4, 4))

# Get the first 5 ground truth indices for this annotation and source image
ground_truth_indices = get_ground_truth_indices(annotation, source_image_idx)[:5]
print("Ground truth indices for this query:", ground_truth_indices)

plot_images(celeba, indices=ground_truth_indices, n_cols=5, n_rows=1, figsize=(10, 2))

### Evaluation Function

We evaluate the retrieval performance of each fusion mechanism on the benchmark dataset, comparing it against the baseline method.

We compute the recall and precision metrics for each source image in the query for `"K = {1, 5, 10}"`.
Then we average the result across all source images and keep track on each query separately.

In [ ]:
def retrieve_topk(scores: torch.Tensor, exclude_idx: int, k: int = 10) -> list[int]:
    """Return the top-k gallery indices by score, excluding the source image.

    Args:
        scores:      (N,) similarity or composite score for every gallery image.
        exclude_idx: index of the source image to remove from results.
        k:           number of results to return.
    Returns:
        List of up to k gallery indices, ranked by descending score.
    """
    _, topk = torch.topk(scores, k=k + 1)
    return topk[topk != exclude_idx][:k].tolist()


def evaluate(
    annotations: list[dict],
    make_scorer: Callable,
    verbose: bool = True,
) -> list[dict]:
    """Evaluate retrieval performance on the benchmark.

    Single driver for all methods. Each method supplies a *scorer factory*:
        make_scorer(annotation) -> scorer(source_idx) -> (N,) gallery scores.

    Per-query work (z-scoring, prompt tokenisation, constraint vectors) happens
    once inside ``make_scorer``; the inner ``scorer`` loop is then fast.

    Args:
        annotations: list of benchmark annotations (loaded from the JSON file).
        make_scorer: callable — given an annotation returns
                     ``scorer(source_idx) -> (N,) score tensor``.
        verbose:     whether to print per-query progress.
    Returns:
        list[ dict[ source_idx -> dict[ k -> metrics_dict ] ] ]
    """
    results = []
    for i, ann in enumerate(annotations):
        if verbose:
            print(f"Evaluating query Q{i+1}: {get_text_query(ann)}")
        scorer = make_scorer(ann)   # per-query setup executed once
        per_source = {}
        for src in get_source_image_idxs(ann):
            retrieved = retrieve_topk(scorer(src), exclude_idx=src, k=10)
            per_source[src] = {
                k: evaluate_retrieval(retrieved, get_ground_truth_indices(ann, src), k)
                for k in (1, 5, 10)
            }
        results.append(per_source)
    return results


def compute_query_average_results(query_evaluation_results: dict) -> dict:
    '''
    Computes the average Recall@K and Precision@K across all source images of a query for K=1, 5, and 10.
    Args:
    - query_evaluation_results: A dict mapping source image index to its evaluation metrics for K=1, 5, and 10.
    Returns:
    - A dictionary containing the average results for the query
    '''
    average_results = {}

    for k in [1, 5, 10]:
        # Compute the sum of Recall@K and Precision@K across all source images for this query
        recall_sum = 0
        precision_sum = 0
        # Get the number of test that we run for this query
        num_images = len(query_evaluation_results)

        # Iterate over the evaluation results for each source image and accumulate the Recall@K and Precision@K values
        for _, eval_metrics_per_k in query_evaluation_results.items():
            recall_sum += eval_metrics_per_k[k][f"Recall@{k}"]
            precision_sum += eval_metrics_per_k[k][f"Precision@{k}"]
        # Compute the average Recall@K and Precision@K for this query and store it in the average_results dictionary        
        average_results[f"Recall@{k}"] = recall_sum / num_images
        average_results[f"Precision@{k}"] = precision_sum / num_images

        # Compute the 95% confidence intervals for Recall@K and Precision@K
        recall_std_error = np.sqrt((average_results[f"Recall@{k}"] * (1 - average_results[f"Recall@{k}"])) / num_images)
        precision_std_error = np.sqrt((average_results[f"Precision@{k}"] * (1 - average_results[f"Precision@{k}"])) / num_images)
        average_results[f"Recall@{k}_CI"] = 1.96 * recall_std_error
        average_results[f"Precision@{k}_CI"] = 1.96 * precision_std_error

    return average_results

## Baseline Method
To establish a baseline for our retrieval system, we evaluate a **zero-shot, training-free approach** that relies exclusively on CLIP embeddings and cosine similarity.

The baseline uses simple latent space arithmetic by combining the text and image embeddings, without any learning or explicit alignment.
The query is expressed as the weighted sum of the text and image embeddings, which is then used to find the nearest neighbours in the dataset.

> **Note:** the baseline encodes the full query string (including `+`/`−` characters) as a single CLIP text embedding rather than decomposing it into signed attribute vectors.

### Parameters

In [ ]:
ALPHA = 1.0   # weight for the text embedding in the fusion
BETA  = 1.0   # weight for the image embedding in the fusion

### Scorer

In [ ]:
def baseline_fusion(normalized_text_embedding: list[torch.Tensor], normalized_image_embedding: torch.Tensor, ALPHA: float = 1.0, BETA: float = 1.0) -> torch.Tensor:
    """
    Combine the query text and source image embeddings into a single query embedding.
    Parameters:
        - normalized_text_embedding: A list of normalized text embeddings for each component of the text query
        - normalized_image_embedding: A normalized image embedding for the source image
        - ALPHA: The weight for the text embedding in the fusion mechanism
        - BETA: The weight for the image embedding in the fusion mechanism
    Return:
        - A single query embedding that combines the text and image embeddings.
    """
    if isinstance(normalized_text_embedding, list):
        normalized_text_embedding = torch.stack([embedding.view(-1) for embedding in normalized_text_embedding]).sum(dim=0)
    else:
        normalized_text_embedding = normalized_text_embedding.view(-1)
    return ALPHA * normalized_text_embedding + BETA * normalized_image_embedding.view(-1)


def baseline_embed_query(text_query: str, device) -> list[torch.Tensor]:
    """Encode the full text query into a single normalised 1-D tensor."""
    return [encode_text(text_query, device)]


def baseline_scorer(
    embeddings: torch.Tensor,
    fusion_mechanism: Callable,
    query_embedding_function: Callable,
    alpha: float = 1.0,
    beta: float = 1.0,
) -> Callable:
    """Scorer factory for the fused-query baseline.

    Encodes the text query once per annotation, then for each source image
    returns gallery cosine similarities against the L2-normalised fused embedding:
        score(x) = embeddings[x] · normalise(fusion(text_emb, img_emb))

    Args:
        embeddings:               (N, D) gallery embeddings.
        fusion_mechanism:         function (text, image, ALPHA, BETA) -> fused vector.
        query_embedding_function: function (query_str, device) -> text embedding(s).
        alpha: text weight forwarded to fusion_mechanism.
        beta:  image weight forwarded to fusion_mechanism.
    Returns:
        make_scorer(annotation) -> scorer(source_idx) -> (N,) score tensor.
    """
    def make_scorer(annotation: dict) -> Callable:
        text_emb = query_embedding_function(get_text_query(annotation), device)

        def scorer(source_idx: int) -> torch.Tensor:
            fused = fusion_mechanism(text_emb, embeddings[source_idx], ALPHA=alpha, BETA=beta)
            return embeddings @ F.normalize(fused, dim=0)

        return scorer
    return make_scorer


### Evaluation & Plot

In [ ]:
evaluation_results_baseline = evaluate(
    annotations,
    baseline_scorer(embeddings, baseline_fusion, baseline_embed_query, alpha=ALPHA, beta=BETA),
)
average_results_per_query_baseline = [
    compute_query_average_results(q) for q in evaluation_results_baseline
]

plot_metrics_across_k(average_results_per_query_baseline, title="Baseline Fusion Performance across K")

## Source-Profile Matching (training-free)

The baseline composes raw embeddings — text plus image summed in CLIP space — which carries two structural problems:

1. **Source leakage.** The raw image embedding injects *every* attribute of the source, including the ones the query asks to change, and pulls the ranking toward look-alikes of the source.
2. **Embedding-space negation.** Negating an attribute embedding (`−e⁺`) points at a region of CLIP space that is *not* the linguistic complement of the concept.

Source-Profile Matching keeps the baseline's ingredients (text similarities + image similarity) but recasts the composition in **per-attribute similarity space**, guided by the benchmark's ground-truth rule (a valid target strictly satisfies the query's +/− constraints and stays within **Hamming distance 2** of the source's 40-bit attribute vector):

1. Encode **one simple text prompt per attribute** and compute the gallery's attribute logit matrix $L \in \mathbb{R}^{N \times 40}$ (cosine of every image against every attribute).
2. **Calibrate** it by z-scoring each column over the gallery — raw CLIP cosines have very different per-attribute means and spreads, so without this step high-mean attributes dominate any sum across columns.
3. Read the **source profile** directly from the source image's own row of the calibrated matrix $Z$.
4. Score every candidate $x$ for source $s$ and signed query $q$ with three terms:

$$\text{score}(x) = w_q \sum_{j \in q} s_j\, Z_{xj} \;-\; w_r \sum_{j \notin q} (Z_{xj} - Z_{sj})^2 \;+\; w_v \cos(e_x, e_s)$$

a **hard-constraint** term on the queried attributes, a **profile-proximity** term on the unqueried ones (this is the Hamming-≤2 budget, measured in calibrated logit space), and an optional raw **visual-similarity** term for identity/pose.

Negation now lives in *calibrated similarity space* (subtracting the z-scored logit) instead of embedding space, and the source contributes **per-attribute agreement** instead of one opaque vector — directly addressing both baseline problems.


### Parameters

In [ ]:
GRID_W_PROFILE = [0.05, 0.1, 0.2, 0.4]   # profile-proximity penalty weight candidates
GRID_W_VISUAL  = [0.0, 0.5, 1.0]          # visual identity weight candidates

### Scorer

The per-attribute embedding bank, then `profile_matching_scorer` and its helpers. **This same scorer is reused unchanged by Prompt Ensembling and CoOp** — only the bank differs, so any performance difference is attributable to the bank alone.

#### Attribute text embeddings

In [ ]:
# One simple text embedding per attribute — the same bare-name prompts as the
# cosine-heatmap diagnostics, so the two views of attribute-text geometry stay
# aligned. This is deliberately the *simplest possible* bank: Prompt Ensembling upgrades
# it while keeping the fusion mechanism fixed.
@torch.no_grad()
def precompute_attribute_text_embeddings() -> torch.Tensor:
    """Return a (n_attrs, D) L2-normalized matrix of CLIP text embeddings, one row per attribute."""
    prompts = [name.replace("_", " ").lower() for name in attr_names]
    return encode_texts(prompts, device)

ATTR_TEXT_EMBS = precompute_attribute_text_embeddings().to(embeddings.device)
print(f"ATTR_TEXT_EMBS: {tuple(ATTR_TEXT_EMBS.shape)}")


#### Profile-matching scorer

In [ ]:
# Profile matching is a *scoring layer*: it consumes a per-attribute
# text-embedding bank and replaces "sum the queried logit columns" with
# "match the candidate's full attribute profile against the source's profile
# with the queried bits flipped".


def compute_attribute_logits(
    embeddings: torch.Tensor,
    E_pos: torch.Tensor,
    E_neg: torch.Tensor | None = None,
) -> torch.Tensor:
    """(N, n_attrs) raw attribute logits. With a pos/neg bank the logit is the
    pos-minus-neg cosine margin; with a single bank it is the plain cosine."""
    logits = embeddings @ E_pos.T
    if E_neg is not None:
        logits = logits - embeddings @ E_neg.T
    return logits


def zscore_columns(logits: torch.Tensor) -> torch.Tensor:
    """Standardise each attribute column over the gallery. CLIP cosines have
    very different per-attribute means/spreads; without this, high-mean
    attributes dominate any sum across columns."""
    mean = logits.mean(dim=0, keepdim=True)
    std = logits.std(dim=0, keepdim=True).clamp_min(1e-6)
    return (logits - mean) / std


def parse_query_signs(text_query: str) -> tuple[list[int], list[int]]:
    """'+Bald, -Eyeglasses' -> ([attribute2idx['Bald']], [attribute2idx['Eyeglasses']])."""
    pos_idx, neg_idx = [], []
    for component in text_query.split(","):
        component = component.strip()
        if not component:
            continue
        sign_char, attr_name = component[0], component[1:].strip()
        j = attribute2idx[attr_name]
        (pos_idx if sign_char == "+" else neg_idx).append(j)
    return pos_idx, neg_idx


def profile_matching_scorer(
    embeddings: torch.Tensor,
    E_pos: torch.Tensor,
    E_neg: torch.Tensor | None = None,
    w_query: float = 1.0,
    w_profile: float = 0.1,
    w_visual: float = 0.0,
) -> Callable:
    """Scorer factory for attribute-profile matching.

    Pre-computes the z-scored attribute logit matrix once, then returns a
    ``make_scorer`` closure that pre-computes the per-query constraint vector
    once, and an inner per-source ``scorer`` closure:

        score(x) = w_query   * Σ_{j ∈ query}  sign_j · Z[x,j]
                 - w_profile * Σ_{j ∉ query}  (Z[x,j] − Z[src,j])²
                 + w_visual  * cos(e_x, e_src)

    Used by Source-Profile Matching, Prompt Ensembling, and CoOp — they differ only in E_pos/E_neg.

    Args:
        embeddings: (N, D) L2-normalised gallery embeddings.
        E_pos:      (n_attrs, D) positive attribute text embeddings.
        E_neg:      (n_attrs, D) negative attribute text embeddings, or None.
        w_query:    weight for the constraint (queried-attribute) term.
        w_profile:  weight for the identity-preservation (profile-proximity) term.
        w_visual:   weight for direct visual cosine similarity.
    Returns:
        make_scorer(annotation) -> scorer(source_idx) -> (N,) score tensor.
    """
    Z = zscore_columns(compute_attribute_logits(embeddings, E_pos, E_neg))  # (N, n_attrs)

    def make_scorer(annotation: dict) -> Callable:
        pos_idx, neg_idx = parse_query_signs(get_text_query(annotation))
        queried   = set(pos_idx + neg_idx)
        unqueried = [j for j in range(Z.shape[1]) if j not in queried]
        constraint = Z[:, pos_idx].sum(dim=1) - Z[:, neg_idx].sum(dim=1)  # (N,)

        def scorer(source_idx: int) -> torch.Tensor:
            z_src   = Z[source_idx]
            profile = ((Z[:, unqueried] - z_src[unqueried]) ** 2).sum(dim=1)
            scores  = w_query * constraint - w_profile * profile
            if w_visual > 0:
                scores = scores + w_visual * (embeddings @ embeddings[source_idx])
            return scores

        return scorer
    return make_scorer


### Evaluation & Plot

The three weights `w_query`, `w_profile`, `w_visual` are training-free hyperparameters tuned with a
deliberately small grid so sensitivity stays visible. They are tuned **once**, using the simple
bare-name attribute bank (Source-Profile Matching); Prompt Ensembling and CoOp reuse the same `PM_WEIGHTS` with their
respective improved banks — making the three methods directly comparable.

In [ ]:
grid_rows = []
for w_p in GRID_W_PROFILE:
    for w_v in GRID_W_VISUAL:
        res = evaluate(
            annotations,
            profile_matching_scorer(embeddings, ATTR_TEXT_EMBS, w_query=1.0, w_profile=w_p, w_visual=w_v),
            verbose=False,
        )
        r10 = mean_recall_at_10(res)
        grid_rows.append((w_p, w_v, r10))
        print(f"w_profile={w_p:<5} w_visual={w_v:<4} mean Recall@10={r10:.4f}")

best_w_profile, best_w_visual, best_r10 = max(grid_rows, key=lambda row: row[2])
print(f"\nBest: w_profile={best_w_profile}, w_visual={best_w_visual} (mean Recall@10={best_r10:.4f})")
PM_WEIGHTS = dict(w_query=1.0, w_profile=best_w_profile, w_visual=best_w_visual)

Final evaluation with the selected `PM_WEIGHTS`, then the per-query metrics plot.

In [ ]:
evaluation_results_profmatch = evaluate(
    annotations,
    profile_matching_scorer(embeddings, ATTR_TEXT_EMBS, **PM_WEIGHTS),
    verbose=False,
)
average_results_per_query_profmatch = [
    compute_query_average_results(q) for q in evaluation_results_profmatch
]
print(f"Source-Profile Matching: mean Recall@10 = {mean_recall_at_10(evaluation_results_profmatch):.4f}")

plot_metrics_across_k(
    average_results_per_query_profmatch,
    title="Source-Profile Matching — Performance across K",
)

## Prompt Ensembling (training-free)

Source-Profile Matching fixed the **fusion mechanism**; its remaining weakness is the **text embeddings themselves** — a single bare prompt per attribute is a noisy estimate of the concept. Prompt Ensembling keeps the scoring layer **and the very same weights** frozen, and upgrades only the per-attribute bank:

- **3a. Expanded template bank.** Each attribute's positive embedding `e⁺` is the ensemble of several person-referring phrases run through CLIP's full 80-template ImageNet prompt set plus a handful of portrait-specific templates.
- **3b. Linguistic negatives.** A separate `e⁻` is built from real negative descriptions ("a person without {attr}", "a clean-shaven person", ...). The attribute logit becomes the **pos-minus-neg margin** `cos(x, e⁺) − cos(x, e⁻)` — a stronger, noise-cancelled signal than a single positive cosine, and a real linguistic complement instead of a sign flip.

Because the fusion mechanism and its weights are inherited unchanged from Source-Profile Matching, any improvement here is attributable to the embedding bank alone.


### Parameters

Prompt Ensembling has no parameters of its own — it reuses `PM_WEIGHTS` from Source-Profile Matching. Only the phrase/template banks below change.

### Scorer

Only the bank changes: Prompt Ensembling builds an **ensembled** pos/neg bank and feeds it to **Source-Profile Matching's `profile_matching_scorer`, unchanged**. Any gain over Source-Profile Matching is therefore attributable to the embedding bank alone.

#### Attribute phrase banks

In [ ]:
# Person-referring positive AND negative phrases for each CelebA attribute.
# The previous version only stored positives and negated their embedding by -1;
# we now also store linguistic negatives so the negative side of the score is
# computed against an actual "without ..." description.
humanized_mappings_pos = {
    "5_o_Clock_Shadow":     ["a person with a 5 o'clock shadow", "a person with light facial stubble", "a person with short beard stubble", "a face with a 5 o'clock shadow", "a man with a 5 o'clock shadow", "a person with visible beard stubble"],
    "Arched_Eyebrows":      ["a person with arched eyebrows", "a person with curved eyebrows", "a face with high arched eyebrows", "a portrait with strongly arched eyebrows", "a person whose eyebrows are clearly arched"],
    "Attractive":           ["an attractive person", "a good-looking person", "a visually appealing person", "a beautiful person", "an attractive face", "a strikingly attractive person"],
    "Bags_Under_Eyes":      ["a person with bags under the eyes", "a person with eye bags", "a tired-looking person with under-eye bags", "a face with visible under-eye bags", "a portrait of a person with bags under the eyes"],
    "Bald":                 ["a bald person", "a person with no hair", "a person with a shaved head", "a person with a completely bald head", "a man who is bald", "a portrait of a bald person"],
    "Bangs":                ["a person with bangs", "a person with fringe hair", "a face with bangs across the forehead", "a portrait of someone with bangs", "a person whose hair has bangs"],
    "Big_Lips":             ["a person with full lips", "a person with big lips", "a face with prominent lips", "a portrait of a person with very full lips"],
    "Big_Nose":             ["a person with a big nose", "a person with a large nose", "a face with a prominent nose", "a portrait of a person with a noticeably big nose"],
    "Black_Hair":           ["a person with black hair", "a person with dark black hair", "a portrait of a black-haired person", "a person whose hair is black"],
    "Blond_Hair":           ["a person with blond hair", "a person with blonde hair", "a person with light blonde hair", "a portrait of a blond person", "a person whose hair is blonde"],
    "Blurry":               ["a blurry photo of a person", "an out-of-focus image of a person", "a blurred image of a person", "a low-quality blurry portrait", "a defocused photograph of a face"],
    "Brown_Hair":           ["a person with brown hair", "a person with dark brown hair", "a portrait of a brown-haired person", "a person whose hair is brown"],
    "Bushy_Eyebrows":       ["a person with bushy eyebrows", "a person with thick eyebrows", "a face with very thick eyebrows", "a portrait of a person with bushy eyebrows"],
    "Chubby":               ["a chubby person", "a person with a round face", "a person with a chubby face", "a portrait of a chubby person"],
    "Double_Chin":          ["a person with a double chin", "a person with a noticeable double chin", "a face with a clear double chin", "a portrait of a person with a double chin"],
    "Eyeglasses":           ["a person wearing eyeglasses", "a person wearing glasses", "a person with glasses", "a face with eyeglasses", "a portrait of a person wearing glasses", "a person who wears glasses"],
    "Goatee":               ["a person with a goatee", "a person with a goatee beard", "a man with a goatee", "a portrait of a person with a goatee"],
    "Gray_Hair":            ["a person with gray hair", "a person with grey hair", "a person with silver hair", "a portrait of a gray-haired person", "an older person with gray hair"],
    "Heavy_Makeup":         ["a person wearing heavy makeup", "a person with noticeable makeup", "a person with strong makeup", "a face with heavy makeup", "a portrait of a person wearing heavy makeup"],
    "High_Cheekbones":      ["a person with high cheekbones", "a person with prominent cheekbones", "a face with sharply defined high cheekbones"],
    "Male":                 ["a man", "a male person", "a portrait of a man", "a photograph of a man", "a male face"],
    "Mouth_Slightly_Open":  ["a person with their mouth slightly open", "a person with slightly open lips", "a face with parted lips", "a portrait of a person whose mouth is slightly open"],
    "Mustache":             ["a person with a mustache", "a person with facial hair and a mustache", "a man with a mustache", "a portrait of a person with a mustache"],
    "Narrow_Eyes":          ["a person with narrow eyes", "a person with small eyes", "a face with narrow eyes", "a portrait of a person with narrow eyes"],
    "No_Beard":             ["a clean-shaven person", "a person without a beard", "a person with no facial hair", "a portrait of a clean-shaven person", "a face without any beard"],
    "Oval_Face":            ["a person with an oval face", "a person with an oval-shaped face", "a portrait of a person with an oval face"],
    "Pale_Skin":            ["a person with pale skin", "a person with light skin tone", "a portrait of a person with pale skin", "a face with very pale skin"],
    "Pointy_Nose":          ["a person with a pointy nose", "a person with a sharp nose", "a face with a pointy nose"],
    "Receding_Hairline":    ["a person with a receding hairline", "a person with thinning hairline", "a portrait of a person whose hairline is receding"],
    "Rosy_Cheeks":          ["a person with rosy cheeks", "a person with flushed cheeks", "a face with rosy cheeks"],
    "Sideburns":            ["a person with sideburns", "a person with long sideburns", "a face with sideburns"],
    "Smiling":              ["a smiling person", "a person who is smiling", "a person with a happy expression", "a person with a big smile", "a portrait of a smiling person", "a face with a smile"],
    "Straight_Hair":        ["a person with straight hair", "a person with smooth straight hair", "a portrait of a person with straight hair"],
    "Wavy_Hair":            ["a person with wavy hair", "a person with curly wavy hair", "a portrait of a person with wavy hair"],
    "Wearing_Earrings":     ["a person wearing earrings", "a person with earrings", "a portrait of a person wearing earrings"],
    "Wearing_Hat":          ["a person wearing a hat", "a person with a hat", "a portrait of a person wearing a hat"],
    "Wearing_Lipstick":     ["a person wearing lipstick", "a person with lipstick", "a portrait of a person wearing lipstick"],
    "Wearing_Necklace":     ["a person wearing a necklace", "a person with a necklace", "a portrait of a person wearing a necklace"],
    "Wearing_Necktie":      ["a person wearing a necktie", "a person with a tie", "a portrait of a person wearing a necktie"],
    "Young":                ["a young person", "a youthful person", "a person who looks young", "a portrait of a young person", "a young-looking face"],
}

# Linguistic negatives. We avoid the "not X" construction wherever possible because
# CLIP's text encoder attends to the object token regardless of the "not" — phrasing
# matters. Where a clean linguistic opposite exists (e.g. clean-shaven vs bearded)
# we use it; otherwise we lean on "without {attr}" / "no {attr}" framings.
humanized_mappings_neg = {
    "5_o_Clock_Shadow":     ["a clean-shaven person", "a person with no facial stubble", "a person without a 5 o'clock shadow", "a smoothly shaven face"],
    "Arched_Eyebrows":      ["a person with flat eyebrows", "a person whose eyebrows are not arched", "a face with straight eyebrows"],
    "Attractive":           ["an unattractive person", "a plain-looking person", "an ordinary-looking person"],
    "Bags_Under_Eyes":      ["a person without bags under the eyes", "a person with no eye bags", "a fresh-looking face without under-eye bags"],
    "Bald":                 ["a person with hair", "a person with a full head of hair", "a person who is not bald"],
    "Bangs":                ["a person without bangs", "a person with no fringe", "a face without bangs"],
    "Big_Lips":             ["a person with thin lips", "a person with small lips", "a person without big lips"],
    "Big_Nose":             ["a person with a small nose", "a person without a big nose", "a face with a small nose"],
    "Black_Hair":           ["a person without black hair", "a person whose hair is not black"],
    "Blond_Hair":           ["a person without blond hair", "a person whose hair is not blonde"],
    "Blurry":               ["a sharp clear photo of a person", "a high quality in-focus portrait", "a crisp clear image of a face"],
    "Brown_Hair":           ["a person without brown hair", "a person whose hair is not brown"],
    "Bushy_Eyebrows":       ["a person with thin eyebrows", "a person without bushy eyebrows"],
    "Chubby":               ["a thin person", "a person with a slim face", "a person who is not chubby"],
    "Double_Chin":          ["a person without a double chin", "a person with a defined jawline"],
    "Eyeglasses":           ["a person without eyeglasses", "a person not wearing glasses", "a face without glasses", "a person with bare eyes"],
    "Goatee":               ["a person without a goatee", "a clean-shaven person", "a person with no goatee"],
    "Gray_Hair":            ["a person without gray hair", "a person whose hair is not gray"],
    "Heavy_Makeup":         ["a person with no makeup", "a person without makeup", "a face without heavy makeup", "a person with a bare natural face"],
    "High_Cheekbones":      ["a person without high cheekbones", "a person with flat cheeks"],
    "Male":                 ["a woman", "a female person", "a portrait of a woman", "a female face"],
    "Mouth_Slightly_Open":  ["a person with a closed mouth", "a person with closed lips", "a person whose mouth is shut"],
    "Mustache":             ["a clean-shaven person", "a person without a mustache", "a person with no mustache"],
    "Narrow_Eyes":          ["a person with wide eyes", "a person with big eyes", "a person without narrow eyes"],
    "No_Beard":             ["a person with a beard", "a bearded person", "a person with facial hair"],
    "Oval_Face":            ["a person without an oval face", "a person with a round face", "a person with a square face"],
    "Pale_Skin":            ["a person with dark skin", "a person with a tanned complexion", "a person without pale skin"],
    "Pointy_Nose":          ["a person with a rounded nose", "a person without a pointy nose"],
    "Receding_Hairline":    ["a person with a full hairline", "a person without a receding hairline"],
    "Rosy_Cheeks":          ["a person without rosy cheeks", "a person with pale cheeks"],
    "Sideburns":            ["a clean-shaven person", "a person without sideburns"],
    "Smiling":              ["a person with a neutral expression", "a person who is not smiling", "a serious-looking person", "a person with a straight face"],
    "Straight_Hair":        ["a person with curly hair", "a person without straight hair"],
    "Wavy_Hair":            ["a person with straight hair", "a person without wavy hair"],
    "Wearing_Earrings":     ["a person without earrings", "a person not wearing any earrings"],
    "Wearing_Hat":          ["a person without a hat", "a person not wearing a hat", "a bare-headed person"],
    "Wearing_Lipstick":     ["a person without lipstick", "a person with bare lips"],
    "Wearing_Necklace":     ["a person without a necklace", "a person with a bare neck"],
    "Wearing_Necktie":      ["a person without a necktie", "a person with an open collar"],
    "Young":                ["an old person", "an elderly person", "an older person", "a senior person"],
}

#### CLIP ImageNet prompt templates

In [ ]:
# CLIP's official ImageNet 80-template set (the canonical zero-shot ensemble) plus a
# small number of portrait-specific templates appropriate to CelebA.
clip_imagenet_templates = [
    "a bad photo of {phrase}.", "a photo of many {phrase}.", "a sculpture of {phrase}.",
    "a photo of the hard to see {phrase}.", "a low resolution photo of {phrase}.",
    "a rendering of {phrase}.", "graffiti of {phrase}.", "a bad photo of {phrase}.",
    "a cropped photo of {phrase}.", "a tattoo of {phrase}.", "the embroidered {phrase}.",
    "a photo of a hard to see {phrase}.", "a bright photo of {phrase}.", "a photo of a clean {phrase}.",
    "a photo of a dirty {phrase}.", "a dark photo of {phrase}.", "a drawing of {phrase}.",
    "a photo of my {phrase}.", "the plastic {phrase}.", "a photo of the cool {phrase}.",
    "a close-up photo of {phrase}.", "a black and white photo of {phrase}.", "a painting of {phrase}.",
    "a painting of {phrase}.", "a pixelated photo of {phrase}.", "a sculpture of {phrase}.",
    "a bright photo of {phrase}.", "a cropped photo of {phrase}.", "a plastic {phrase}.",
    "a photo of the dirty {phrase}.", "a jpeg corrupted photo of {phrase}.",
    "a blurry photo of {phrase}.", "a photo of {phrase}.", "a good photo of {phrase}.",
    "a rendering of {phrase}.", "a {phrase} in a video game.", "a photo of one {phrase}.",
    "a doodle of {phrase}.", "a close-up photo of {phrase}.", "a photo of {phrase}.",
    "the origami {phrase}.", "a sketch of {phrase}.", "a doodle of {phrase}.",
    "a origami {phrase}.", "a low resolution photo of {phrase}.", "the toy {phrase}.",
    "a rendition of {phrase}.", "a photo of the clean {phrase}.", "a photo of a large {phrase}.",
    "a rendition of {phrase}.", "a photo of a nice {phrase}.", "a photo of a weird {phrase}.",
    "a blurry photo of {phrase}.", "a cartoon {phrase}.", "art of {phrase}.",
    "a sketch of {phrase}.", "a embroidered {phrase}.", "a pixelated photo of {phrase}.",
    "itap of {phrase}.", "a jpeg corrupted photo of {phrase}.", "a good photo of {phrase}.",
    "a plushie {phrase}.", "a photo of the nice {phrase}.", "a photo of the small {phrase}.",
    "a photo of the weird {phrase}.", "the cartoon {phrase}.", "art of {phrase}.",
    "a drawing of {phrase}.", "a photo of the large {phrase}.", "a black and white photo of {phrase}.",
    "the plushie {phrase}.", "a dark photo of {phrase}.", "itap of {phrase}.",
    "graffiti of {phrase}.", "a toy {phrase}.", "itap of {phrase}.",
    "a photo of a cool {phrase}.", "a photo of a small {phrase}.", "a tattoo of {phrase}.",
]
portrait_templates = [
    "a portrait of {phrase}.",
    "a portrait photograph of {phrase}.",
    "a closeup headshot of {phrase}.",
    "a candid photo of {phrase}.",
    "a studio portrait of {phrase}.",
    "a high-resolution headshot of {phrase}.",
    "a face photo of {phrase}.",
    "a photo showing the face of {phrase}.",
    "a frontal photo of {phrase}.",
    "a clear photo of {phrase}.",
]
prompt_templates_v2 = clip_imagenet_templates + portrait_templates


@torch.no_grad()
def _encode_phrases_through_templates(phrases: list[str], templates: list[str]) -> torch.Tensor:
    """Encode every (phrase x template) pair, L2-normalize each, mean-pool, re-normalize.

    Batched in a single processor/model call for speed.
    """
    prompts = [template.format(phrase=phrase) for phrase in phrases for template in templates]
    embs = encode_texts(prompts, device)   # (P, D), per-row normalized
    mean_emb = embs.mean(dim=0)
    return mean_emb / mean_emb.norm()


@torch.no_grad()
def precompute_attribute_pos_neg_embeddings() -> tuple[torch.Tensor, torch.Tensor]:
    """Return (E_pos, E_neg), each (40, 512) and L2-normalized.

    E_pos[i] = ensemble over (positive phrases for attribute i) x (templates)
    E_neg[i] = ensemble over (negative phrases for attribute i) x (templates)
    """
    pos_embs, neg_embs = [], []
    for name in attr_names:
        pos_embs.append(_encode_phrases_through_templates(humanized_mappings_pos[name], prompt_templates_v2))
        neg_embs.append(_encode_phrases_through_templates(humanized_mappings_neg[name], prompt_templates_v2))
    E_pos = torch.stack(pos_embs, dim=0)
    E_neg = torch.stack(neg_embs, dim=0)
    return E_pos, E_neg


print("Precomputing pos/neg attribute embeddings with the expanded template bank (this may take a minute)...")
E_POS, E_NEG = precompute_attribute_pos_neg_embeddings()
E_POS = E_POS.to(embeddings.device)
E_NEG = E_NEG.to(embeddings.device)
print(f"E_POS: {tuple(E_POS.shape)},  E_NEG: {tuple(E_NEG.shape)}")

# Calibrated logits from the ensemble bank, reused by the Pic2Word hybrid.
Z_ENS = zscore_columns(compute_attribute_logits(embeddings, E_POS, E_NEG))

### Evaluation & Plot

In [ ]:
# Same scoring layer and weights as Source-Profile Matching — only the embedding bank changes.
evaluation_results_promptens = evaluate(
    annotations,
    profile_matching_scorer(embeddings, E_POS, E_NEG, **PM_WEIGHTS),
    verbose=False,
)
average_results_per_query_promptens = [
    compute_query_average_results(q) for q in evaluation_results_promptens
]
print(f"Prompt Ensembling: mean Recall@10 = {mean_recall_at_10(evaluation_results_promptens):.4f}")


plot_methods_comparison(
    {
        "Baseline":                            average_results_per_query_baseline,
        "Source-Profile Matching":                average_results_per_query_profmatch,
        "Prompt Ensembling":  average_results_per_query_promptens,
    },
    title="Training-Free Method Comparison — per-query Recall@K and Precision@K",
)

# Training-Based Method — CoOp

Up to now every method is training-free: the CLIP weights and the prompts are fixed. **CoOp** ([Context Optimization, Zhou et al. 2022](https://arxiv.org/abs/2109.01134)) keeps CLIP entirely frozen but replaces the *handcrafted prefix* in each prompt with a small set of **learnable context tokens** — `M = 16` vectors in the same embedding space as CLIP's word embeddings.

### How we frame it for CelebA's compositional queries
CelebA has 40 binary attributes — every image is annotated with a 40-dim 0/1 label vector — so the natural objective is **multi-label binary classification**. For every attribute `a` we build two prompts:
* **positive:** `[ctx_1] … [ctx_M] a person with {attr_a}.`
* **negative:** `[ctx_1] … [ctx_M] a person without {attr_a}.`

The same `M` learnable context vectors are **shared across all 40 attributes** and across the positive/negative prompts (the canonical "shared context" CoOp variant). With `M = 16`, that is only `M · 512 ≈ 8 k` trainable parameters.

The per-attribute logit is
$$\ell_a(\mathbf{x}) = \cos\!\big(\mathbf{e}_{\text{img}}(\mathbf{x}),\, \mathbf{e}_+^{(a)}\big) - \cos\!\big(\mathbf{e}_{\text{img}}(\mathbf{x}),\, \mathbf{e}_-^{(a)}\big)$$
(scaled by CLIP's `logit_scale`), and we train with **binary cross-entropy** against the multi-label ground truth.

### Test-time retrieval
After training, `e_+^{(a)}` and `e_-^{(a)}` are 40 learned text embeddings. Retrieval feeds them into the **same profile-matching scorer used by Source-Profile Matching and Prompt Ensembling** with the same `PM_WEIGHTS`:
$$
\text{score}(x, q) = w_{\text{query}} \cdot \sum_{j \in q} \text{sign}_j \cdot Z[x,j]
  \;-\; w_{\text{profile}} \cdot \sum_{j \notin q} (Z[x,j] - Z[\text{src},j])^2
  \;+\; w_{\text{visual}} \cdot \cos(e_x, e_{\text{src}})
$$
This means **Methods 2, 3, and CoOp share the exact same evaluation harness** — the only variable is which embedding bank is plugged in.

### Compute budget
The image encoder is frozen throughout. We pre-extract CLIP image features once and train on the **full CelebA train split** (~162 k images) for `COOP_EPOCHS` epochs with AMP on GPU.

### Parameters

In [ ]:
COOP_M        = 16                        # number of learnable context vectors
COOP_EPOCHS   = 20                        # training epochs
COOP_BATCH    = 256                       # mini-batch size
COOP_LR       = 2e-3                      # SGD learning rate
COOP_VAL_FRAC = 0.05                      # fraction of training set held out for val
COOP_USE_AMP  = (device == "cuda")        # automatic mixed precision (requires GPU)

### Training

#### Step 1 — Load the CelebA training split and pre-extract image features

We load the **train** split of CelebA (the previous cells only loaded `test`) and pre-extract every training image's CLIP image embedding **once**, caching to Drive at `clip_cache/train_embeddings.pt`. After this cell, training never touches the CLIP image encoder again.

The torchvision Google-Drive download for CelebA is unreliable (the Drive quota frequently returns HTTP 429/404). If the standard `download=True` path fails, the recommended fallback is to use the same `celeba.zip` already staged on Drive for the test split — many such bundles ship `list_eval_partition.txt` and the `img_align_celeba/` folder, so they already contain the train images. Set `CELEBA_TRAIN_ROOT` below to wherever your CelebA root lives.

In [ ]:
CELEBA_TRAIN_ROOT = Path("/content/datasets")
TRAIN_FEATS_PATH  = Path(EVALUATION_CACHE_DIR) / "train_embeddings.pt"

# Step 1a — load the full CelebA train split
print(f"Loading CelebA train split from {CELEBA_TRAIN_ROOT} ...")
celeba_train = CelebA(root=CELEBA_TRAIN_ROOT, split="train", download=False)
print(f"CelebA train split size: {len(celeba_train)}")

# Step 1b — pre-extract image features once and cache (shared utility, see encoding utilities)
train_features, train_labels = get_encoded_dataset(
    celeba_train, device, str(TRAIN_FEATS_PATH), batch_size=128
)
print(f"train_features dtype: {train_features.dtype}, device: {train_features.device}")

#### Step 2 — Define the CoOp prompt learner

This is the only `nn.Module` with trainable parameters in the entire notebook. The CLIP image and text encoders are kept frozen; only the `M × 512` context matrix is updated by SGD.

**Implementation detail.** We construct each prompt's token IDs once (with `M` placeholder `"X"` tokens in the prefix), embed them via CLIP's token-embedding lookup, then in the forward pass we **overwrite** the `M` prefix-position embeddings with our learnable `ctx` parameter before running the text transformer manually. We pool at the EOS position (CLIP's text-feature convention) and project to the joint embedding space.

In [ ]:
# Note: CoOp does NOT route through encode_texts(). It operates one level below
# the encoder API — it overwrites token embeddings at positions [1:1+M] with the
# learnable `self.ctx` parameter, then calls the shared `_clip_encode_token_emb`
# helper (see encoding utilities section).


class CoOpPromptLearner(nn.Module):
    """Shared-context CoOp on top of HuggingFace CLIP.

    For each attribute we build a positive and a negative prompt of the form
        [SOS] [ctx_1] … [ctx_M] a person with/without {attr}. [EOS] [PAD] …
    The token-embedding lookup happens once at construction time; the forward
    pass replaces positions [1:1+M] with `self.ctx` before running the frozen
    text transformer via `_clip_encode_token_emb`.
    """

    def __init__(self, clip_model: CLIPModel, processor: CLIPProcessor, attr_names: list[str], M: int = 16):
        super().__init__()
        self.clip = clip_model
        self.M    = M
        self.attr_names = attr_names
        self.num_attrs  = len(attr_names)

        embedding_dim = clip_model.text_model.embeddings.token_embedding.embedding_dim

        ctx = torch.empty(M, embedding_dim)
        nn.init.normal_(ctx, std=0.02)
        self.ctx = nn.Parameter(ctx)

        tokenizer = processor.tokenizer
        max_len   = clip_model.config.text_config.max_position_embeddings
        placeholder = " ".join(["X"] * M)

        readable    = [name.replace("_", " ").lower() for name in attr_names]
        pos_prompts = [f"{placeholder} a person with {name}."    for name in readable]
        neg_prompts = [f"{placeholder} a person without {name}." for name in readable]

        pos = tokenizer(pos_prompts, padding="max_length", max_length=max_len, truncation=True, return_tensors="pt")
        neg = tokenizer(neg_prompts, padding="max_length", max_length=max_len, truncation=True, return_tensors="pt")
        self.register_buffer("pos_ids", pos["input_ids"])
        self.register_buffer("neg_ids", neg["input_ids"])

        for p in self.clip.parameters():
            p.requires_grad_(False)

    def _build_token_emb(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Embed prompt tokens and overwrite prefix positions with learnable context."""
        bsz       = input_ids.shape[0]
        token_emb = self.clip.text_model.embeddings.token_embedding(input_ids)
        return torch.cat(
            [
                token_emb[:, :1, :],
                self.ctx.unsqueeze(0).expand(bsz, -1, -1),
                token_emb[:, 1 + self.M:, :],
            ],
            dim=1,
        )

    def _encode(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Build token embeddings with ctx overwrite, then run frozen transformer."""
        ids       = input_ids.to(self.ctx.device)
        token_emb = self._build_token_emb(ids)
        return _clip_encode_token_emb(self.clip, token_emb, ids)

    def forward(self):
        return self._encode(self.pos_ids), self._encode(self.neg_ids)


# Sanity check
coop = CoOpPromptLearner(clip_model=_model, processor=_processor, attr_names=get_attribute_names(celeba), M=COOP_M).to(device)
with torch.no_grad():
    e_pos, e_neg = coop()
print(f"e_pos: {tuple(e_pos.shape)}, e_neg: {tuple(e_neg.shape)}")
n_trainable = sum(p.numel() for p in coop.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_trainable} (should be ~{COOP_M*512})")


#### Step 3 — Training loop

We treat each of the 40 binary attributes as an independent binary classifier whose logit is `cos(img, e+) − cos(img, e−)` scaled by CLIP's `logit_scale`. The loss is **binary cross-entropy with logits** over the multi-label target.

We hold out 5% of the training set as a validation set and track macro-AUC across epochs to pick the best context.

In [ ]:
# ----- Build train / val splits -----
torch.manual_seed(42)
np.random.seed(42)

train_labels_dev = (
    train_labels.to(device)
    if isinstance(train_labels, torch.Tensor)
    else torch.as_tensor(train_labels, device=device)
)

# Full train split (parameters set in CoOp Parameters cell above).
train_feats_sel  = train_features
train_labels_sel = train_labels_dev

N        = train_feats_sel.shape[0]
perm     = torch.randperm(N, device=device)
val_size = max(1, int(round(COOP_VAL_FRAC * N)))
val_idx  = perm[:val_size]
trn_idx  = perm[val_size:]
X_tr, Y_tr = train_feats_sel[trn_idx], train_labels_sel[trn_idx].float()
X_va, Y_va = train_feats_sel[val_idx], train_labels_sel[val_idx].float()

print(f"CoOp  |  train: {tuple(X_tr.shape)}  val: {tuple(X_va.shape)}")
print(f"epochs={COOP_EPOCHS}  batch={COOP_BATCH}  lr={COOP_LR}  M={COOP_M}  amp={COOP_USE_AMP}")

coop = CoOpPromptLearner(
    clip_model=_model, processor=_processor,
    attr_names=get_attribute_names(celeba), M=COOP_M,
).to(device)
optimizer   = torch.optim.SGD([coop.ctx], lr=COOP_LR, momentum=0.9)
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=COOP_EPOCHS)
scaler      = torch.cuda.amp.GradScaler(enabled=COOP_USE_AMP)
logit_scale = _model.logit_scale.exp().detach()


@torch.no_grad()
def macro_auc(logits: torch.Tensor, targets: torch.Tensor) -> float:
    """Macro-averaged AUC over the 40 attributes, computed on `logits.device`."""
    logits  = logits.detach()
    targets = targets.detach()
    aucs: list[torch.Tensor] = []
    n = targets.shape[0]
    for j in range(targets.shape[1]):
        y     = targets[:, j]
        n_pos = int(y.sum().item())
        if n_pos == 0 or n_pos == n:
            continue
        n_neg  = n - n_pos
        order  = torch.argsort(logits[:, j])
        ranks  = torch.empty_like(order, dtype=torch.float32)
        ranks[order] = torch.arange(n, device=order.device, dtype=torch.float32)
        pos_ranks = ranks[y == 1].sum()
        aucs.append((pos_ranks - n_pos * (n_pos - 1) / 2) / (n_pos * n_neg))
    if not aucs:
        return float("nan")
    return float(torch.stack(aucs).mean())


best_val = -1.0
best_ctx = coop.ctx.detach().clone()
n_train  = X_tr.shape[0]

print("Starting CoOp training...")
for epoch in range(COOP_EPOCHS):
    coop.train()
    perm_e     = torch.randperm(n_train, device=device)
    total_loss = 0.0
    for start in range(0, n_train, COOP_BATCH):
        idx = perm_e[start:start + COOP_BATCH]
        x, y = X_tr[idx], Y_tr[idx]
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=COOP_USE_AMP):
            e_pos, e_neg = coop()
            logits = logit_scale * (x @ e_pos.T - x @ e_neg.T)
            loss   = F.binary_cross_entropy_with_logits(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += float(loss.detach()) * x.shape[0]
    scheduler.step()

    coop.eval()
    with torch.no_grad():
        e_pos, e_neg = coop()
        val_logits   = logit_scale * (X_va @ e_pos.T - X_va @ e_neg.T)
        val_auc      = macro_auc(val_logits, Y_va)

    print(f"Epoch {epoch+1:3d}/{COOP_EPOCHS}  loss={total_loss/max(1,n_train):.4f}  val_macroAUC={val_auc:.4f}")
    if val_auc > best_val:
        best_val = val_auc
        best_ctx = coop.ctx.detach().clone()

coop.ctx.data.copy_(best_ctx)
print(f"Best validation macro-AUC: {best_val:.4f}")

COOP_CKPT = Path(EVALUATION_CACHE_DIR) / "coop_ctx.pt"
torch.save({"ctx": best_ctx.cpu(), "M": COOP_M, "val_macro_auc": best_val}, COOP_CKPT)
print(f"Saved context to {COOP_CKPT}")


### Scorer

The learned banks `E_POS_COOP`/`E_NEG_COOP` come from the trained context; they are then fed to **Source-Profile Matching's `profile_matching_scorer` with the same `PM_WEIGHTS`** — an identical harness to Source-Profile Matching and Prompt Ensembling, so the only difference is the *learned* bank.

In [ ]:
coop.eval()
with torch.no_grad():
    E_POS_COOP, E_NEG_COOP = coop()
E_POS_COOP = E_POS_COOP.to(embeddings.device)
E_NEG_COOP = E_NEG_COOP.to(embeddings.device)


### Evaluation & Plot

In [ ]:
evaluation_results_coop = evaluate(
    annotations,
    profile_matching_scorer(embeddings, E_POS_COOP, E_NEG_COOP, **PM_WEIGHTS),
    verbose=False,
)
average_results_per_query_coop = [
    compute_query_average_results(q) for q in evaluation_results_coop
]
print(f"CoOp (M={COOP_M}): mean Recall@10 = {mean_recall_at_10(evaluation_results_coop):.4f}")
plot_metrics_across_k(
    average_results_per_query_coop,
    title=f"CoOp (M={COOP_M}) — Performance across K",
)

# Training-Based Method — Conditional Combiner

The combiner is a small fusion network that *learns* how the source image and the textual conditions should interact, instead of hand-designing the interaction (baseline sum, logit composition, profile matching). It directly targets the project requirements: it **natively accepts multiple textual conditions alongside a visual reference**, **dynamically weighs** them via attention, and fuses **beyond naïve concatenation** via a gated residual.

### Triplet synthesis
The benchmark's generative process is known — a target must strictly satisfy the query's +/− constraints and lie within Hamming distance 2 of the source's attribute vector — so we can replay exactly that process on the **train** split to synthesize `(source, query, target)` triplets:

1. Sample a train image `s` with label vector `a`.
2. Sample a query of 1–3 attributes; the sign is chosen so the query *changes* the source (`+attr` where `a_j = 0`, `-attr` where `a_j = 1`), matching the benchmark's query style.
3. Valid targets are the train images that strictly satisfy the query and are within Hamming ≤ 2 of the flipped vector — a few vectorized boolean ops on the label matrix. One is sampled per `(s, q)`.

Queries are sampled at random, so the combiner never sees the benchmark's specific query list during training. Everything runs on the cached train embeddings; CLIP itself is never touched.

### Architecture (≈1.6M parameters)
Each query term `±attr` is embedded as the corresponding row of the prompt-ensemble bank (`E⁺` for `+attr`, `E⁻` for `-attr`) plus a **learned sign embedding** — this is where positive and negative constraints get distinct, learned semantics in embedding space. The source image embedding acts as the **attention query** over the set of condition embeddings (dynamic per-source weighting), and the pooled conditions enter through a **gated residual** on top of the image embedding:

$$\mathbf{q} = \mathrm{norm}\big(\, e_{img} + g \odot \mathrm{MLP}([\,e_{img}\,;\,\mathrm{pooled}\,]) \,\big), \qquad g = \sigma(W[\,e_{img}\,;\,\mathrm{pooled}\,])$$

### Loss
InfoNCE against the target's image embedding with in-batch negatives, using CLIP's frozen `logit_scale` as temperature. A batch can occasionally contain other valid targets for the same query (false negatives); this is standard contrastive noise and is tolerated.


### Parameters

In [ ]:
MAX_TERMS        = 3         # max attribute conditions per synthetic query
HAMMING_BUDGET   = 2         # max Hamming distance for valid target (matches benchmark)
N_TRAIN_TRIPLETS = 30_000    # number of synthetic training triplets
N_VAL_TRIPLETS   = 2_000     # number of synthetic validation triplets
COMBINER_EPOCHS  = 20        # training epochs
COMBINER_BATCH   = 256       # mini-batch size
COMBINER_LR      = 1e-4      # AdamW learning rate
COMBINER_WD      = 1e-2      # AdamW weight decay
COMBINER_HIDDEN  = 512       # hidden dimension of the gated MLP

### Training

#### Triplet synthesis

In [ ]:
train_labels_bool = (train_labels.to(device) > 0)   # (M, 40) on GPU, for candidate filtering
train_labels_np = train_labels_bool.cpu().numpy()   # CPU copy, for cheap per-sample query sampling
TRAIN_N = train_labels_bool.shape[0]
n_attrs = train_labels_bool.shape[1]


def find_valid_targets(source_labels: torch.Tensor, pos_idx: list[int], neg_idx: list[int]) -> torch.Tensor:
    """Train indices strictly satisfying the query and within HAMMING_BUDGET of
    the source vector with the queried bits flipped — the benchmark's own rule."""
    target = source_labels.clone()
    if pos_idx:
        target[pos_idx] = True
    if neg_idx:
        target[neg_idx] = False
    ok = torch.ones(TRAIN_N, dtype=torch.bool, device=train_labels_bool.device)
    if pos_idx:
        ok &= train_labels_bool[:, pos_idx].all(dim=1)
    if neg_idx:
        ok &= (~train_labels_bool[:, neg_idx]).all(dim=1)
    hamming = (train_labels_bool != target.unsqueeze(0)).sum(dim=1)
    return (ok & (hamming <= HAMMING_BUDGET)).nonzero(as_tuple=True)[0]


def generate_triplet_pool(n_triplets: int, seed: int, log_every: int = 5000):
    """Sample (source_idx, target_idx, cond_attr, cond_sign) rows.

    cond_attr / cond_sign are fixed-width (MAX_TERMS,) rows; sign 0 marks padding.
    """
    rng = np.random.default_rng(seed)
    src, tgt, cond_attr, cond_sign = [], [], [], []
    while len(src) < n_triplets:
        s = int(rng.integers(0, TRAIN_N))
        a_np = train_labels_np[s]
        n_terms = int(rng.integers(1, MAX_TERMS + 1))
        attrs = [int(j) for j in rng.choice(n_attrs, size=n_terms, replace=False)]
        pos_idx = [j for j in attrs if not a_np[j]]   # the query must *change* the source
        neg_idx = [j for j in attrs if a_np[j]]
        candidates = find_valid_targets(train_labels_bool[s], pos_idx, neg_idx)
        candidates = candidates[candidates != s]
        if candidates.numel() == 0:
            continue
        t = int(candidates[int(rng.integers(0, candidates.numel()))])
        attrs_row = pos_idx + neg_idx
        signs_row = [1] * len(pos_idx) + [-1] * len(neg_idx)
        pad = MAX_TERMS - len(attrs_row)
        src.append(s)
        tgt.append(t)
        cond_attr.append(attrs_row + [0] * pad)
        cond_sign.append(signs_row + [0] * pad)
        if len(src) % log_every == 0:
            print(f"  {len(src)}/{n_triplets} triplets")
    return (
        torch.tensor(src), torch.tensor(tgt),
        torch.tensor(cond_attr), torch.tensor(cond_sign),
    )


print("Generating combiner triplets (one-off, a few minutes)...")
trip_src, trip_tgt, trip_attr, trip_sign = generate_triplet_pool(N_TRAIN_TRIPLETS, seed=0)
val_src, val_tgt, val_attr, val_sign  = generate_triplet_pool(N_VAL_TRIPLETS,   seed=1)
print(f"train triplets: {trip_src.shape[0]}, val triplets: {val_src.shape[0]}")


#### Conditional Combiner architecture

In [ ]:
class ConditionalCombiner(nn.Module):
    """Attention-pooled, gated-residual fusion of a source image embedding with
    a variable-size set of signed attribute conditions.

    Each condition ±attr is embedded as the matching row of the (frozen) pos/neg
    text bank plus a learned sign embedding. The image embedding is the attention
    query over the conditions (dynamic weighting), and the pooled conditions
    modify the image embedding through a gated residual.
    """

    def __init__(self, E_pos: torch.Tensor, E_neg: torch.Tensor, hidden: int = 512):
        super().__init__()
        dim = E_pos.shape[1]
        self.register_buffer("E_pos", E_pos.detach().clone())
        self.register_buffer("E_neg", E_neg.detach().clone())
        self.sign_emb = nn.Parameter(torch.empty(2, dim))   # 0: positive, 1: negative
        nn.init.normal_(self.sign_emb, std=0.02)
        self.cond_proj = nn.Linear(dim, dim)
        self.img_query = nn.Linear(dim, dim)
        self.fuse = nn.Sequential(
            nn.Linear(2 * dim, hidden), nn.ReLU(), nn.Linear(hidden, dim)
        )
        self.gate = nn.Sequential(nn.Linear(2 * dim, dim), nn.Sigmoid())

    def forward(self, img_emb: torch.Tensor, cond_attr: torch.Tensor, cond_sign: torch.Tensor) -> torch.Tensor:
        """img_emb: (B, D) L2-normalized. cond_attr: (B, T) attribute indices.
        cond_sign: (B, T) in {+1, -1, 0}; 0 marks padding. Returns (B, D), L2-normalized."""
        pad_mask = cond_sign == 0                                                  # (B, T)
        base = torch.where(
            (cond_sign > 0).unsqueeze(-1), self.E_pos[cond_attr], self.E_neg[cond_attr]
        )                                                                          # (B, T, D)
        sign_idx = (cond_sign < 0).long()
        conds = self.cond_proj(base + self.sign_emb[sign_idx])                     # (B, T, D)

        # Attention pooling with the image as the query: the model learns how much
        # each textual condition should weigh for this particular source image.
        q = self.img_query(img_emb).unsqueeze(1)                                   # (B, 1, D)
        attn = (q * conds).sum(dim=-1) / math.sqrt(conds.shape[-1])                # (B, T)
        attn = attn.masked_fill(pad_mask, float("-inf")).softmax(dim=-1)
        pooled = (attn.unsqueeze(-1) * conds).sum(dim=1)                           # (B, D)

        h = torch.cat([img_emb, pooled], dim=-1)                                   # (B, 2D)
        out = img_emb + self.gate(h) * self.fuse(h)                                # gated residual
        return F.normalize(out, dim=-1)


combiner = ConditionalCombiner(E_POS, E_NEG, hidden=COMBINER_HIDDEN).to(device)
n_params = sum(p.numel() for p in combiner.parameters() if p.requires_grad)
print(f"Combiner trainable parameters: {n_params:,}")


#### Combiner training

In [ ]:
model, processor = get_CLIP_model()
logit_scale_value = model.logit_scale.exp().detach()

optimizer = torch.optim.AdamW(combiner.parameters(), lr=COMBINER_LR, weight_decay=COMBINER_WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=COMBINER_EPOCHS)

trip_src_dev, trip_tgt_dev = trip_src.to(device), trip_tgt.to(device)
trip_attr_dev, trip_sign_dev = trip_attr.to(device), trip_sign.to(device)
val_src_dev, val_attr_dev, val_sign_dev = val_src.to(device), val_attr.to(device), val_sign.to(device)


@torch.no_grad()
def combiner_val_recall_at_10() -> float:
    """Recall@10 on the held-out triplets, against the full train gallery.

    A retrieved index counts as a hit if it is a *valid* target for the
    triplet's query (benchmark rule), not merely the one sampled target.
    """
    combiner.eval()
    hits = 0
    n_val = val_src_dev.shape[0]
    for start in range(0, n_val, 512):   # memory-chunking only; does not affect results
        sl = slice(start, min(start + 512, n_val))
        q = combiner(train_features[val_src_dev[sl]], val_attr_dev[sl], val_sign_dev[sl])
        sims = q @ train_features.T                                            # (B, M)
        sims[torch.arange(q.shape[0], device=device), val_src_dev[sl]] = -1.0  # drop the source
        top10 = sims.topk(10, dim=1).indices                                   # (B, 10)
        for row in range(q.shape[0]):
            i = start + row
            attrs = val_attr_dev[i].tolist()
            signs = val_sign_dev[i].tolist()
            pos_idx = [a for a, s in zip(attrs, signs) if s > 0]
            neg_idx = [a for a, s in zip(attrs, signs) if s < 0]
            target = train_labels_bool[val_src_dev[i]].clone()
            if pos_idx:
                target[pos_idx] = True
            if neg_idx:
                target[neg_idx] = False
            cand = train_labels_bool[top10[row]]                               # (10, 40)
            ok = torch.ones(10, dtype=torch.bool, device=device)
            if pos_idx:
                ok &= cand[:, pos_idx].all(dim=1)
            if neg_idx:
                ok &= (~cand[:, neg_idx]).all(dim=1)
            ok &= (cand != target.unsqueeze(0)).sum(dim=1) <= HAMMING_BUDGET
            hits += int(ok.any())
    return hits / n_val


best_val_combiner = -1.0
best_combiner_state = None
n_train_trip = trip_src_dev.shape[0]

print("Starting combiner training...")
for epoch in range(COMBINER_EPOCHS):
    combiner.train()
    perm = torch.randperm(n_train_trip, device=device)
    total_loss = 0.0
    for start in range(0, n_train_trip, COMBINER_BATCH):
        idx = perm[start:start + COMBINER_BATCH]
        q = combiner(train_features[trip_src_dev[idx]], trip_attr_dev[idx], trip_sign_dev[idx])
        t = train_features[trip_tgt_dev[idx]]
        logits = logit_scale_value * (q @ t.T)                # in-batch negatives
        labels_ce = torch.arange(q.shape[0], device=device)
        loss = F.cross_entropy(logits, labels_ce)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.detach()) * q.shape[0]
    scheduler.step()

    val_r10 = combiner_val_recall_at_10()
    print(f"Epoch {epoch+1:3d}/{COMBINER_EPOCHS}  loss={total_loss / n_train_trip:.4f}  val Recall@10={val_r10:.4f}")
    if val_r10 > best_val_combiner:
        best_val_combiner = val_r10
        best_combiner_state = {k: v.detach().clone() for k, v in combiner.state_dict().items()}

combiner.load_state_dict(best_combiner_state)
print(f"Best val Recall@10: {best_val_combiner:.4f}")

COMBINER_CKPT = Path(EVALUATION_CACHE_DIR) / "combiner.pt"
torch.save(
    {"state_dict": {k: v.cpu() for k, v in best_combiner_state.items()}, "val_recall10": best_val_combiner},
    COMBINER_CKPT,
)
print(f"Saved combiner to {COMBINER_CKPT}")


### Scorer

In [ ]:
def query_to_condition_rows(text_query: str) -> tuple[torch.Tensor, torch.Tensor]:
    """Benchmark query string -> (cond_attr, cond_sign) tensors of shape (1, T)."""
    pos_idx, neg_idx = parse_query_signs(text_query)
    attrs  = pos_idx + neg_idx
    signs  = [1] * len(pos_idx) + [-1] * len(neg_idx)
    width  = max(MAX_TERMS, len(attrs))
    pad    = width - len(attrs)
    cond_attr = torch.tensor([attrs + [0] * pad], device=embeddings.device)
    cond_sign = torch.tensor([signs + [0] * pad], device=embeddings.device)
    return cond_attr, cond_sign


def combiner_scorer(
    embeddings: torch.Tensor,
    combiner: nn.Module,
) -> Callable:
    """Scorer factory for the Conditional Combiner.

    Encodes the query conditions once per annotation and returns gallery
    cosine similarities against the combiner's output embedding.

    Args:
        embeddings: (N, D) L2-normalised gallery embeddings.
        combiner:   trained ConditionalCombiner.
    Returns:
        make_scorer(annotation) -> scorer(source_idx) -> (N,) score tensor.
    """
    combiner.eval()

    def make_scorer(annotation: dict) -> Callable:
        cond_attr, cond_sign = query_to_condition_rows(get_text_query(annotation))

        @torch.no_grad()
        def scorer(source_idx: int) -> torch.Tensor:
            q = combiner(embeddings[source_idx].unsqueeze(0), cond_attr, cond_sign).squeeze(0)
            return embeddings @ q

        return scorer
    return make_scorer


### Evaluation & Plot

In [ ]:
evaluation_results_combiner = evaluate(
    annotations,
    combiner_scorer(embeddings, combiner),
)
average_results_per_query_combiner = [
    compute_query_average_results(q) for q in evaluation_results_combiner
]
plot_metrics_across_k(
    average_results_per_query_combiner,
    title="Conditional Combiner — Performance across K",
)


# Training-Based Method — Pic2Word

[Pic2Word (Saito et al., 2023)](https://arxiv.org/abs/2302.03084) frames composed retrieval as **textual inversion**: a small mapper network turns the source image embedding into a **pseudo-token** `S*` that lives in CLIP's word-embedding space. The query is then expressed *entirely in language* — `"a photo of S* with eyeglasses without smiling"` — and encoded by the frozen text encoder, which composes the source's visual content with the textual modifications natively.

Two properties make it attractive here:

* **Self-supervised** — training needs *images only*, no triplets and no attribute labels: the mapper is optimized so that `text_encode("a photo of S*")` reconstructs the image embedding, with a symmetric contrastive loss over a batch.
* **We already own the hard part** — injecting a vector at a token position inside the frozen text encoder is exactly the machinery of `CoOpPromptLearner._encode_text_with_ctx`. The only difference: the injected vector is *per sample* (the mapper's output) instead of a shared learned parameter.

### Constraint handling and the hybrid variant
CLIP's text encoder composes attributes only weakly and is especially poor at negation (documented in Prompt Ensembling): `"without eyeglasses"` still activates on the *eyeglasses* token, and even positive attributes barely move the query off the strong pseudo-token `S*`. We therefore evaluate two variants:

1. **Prompt only** — all `+`/`−` attributes are expressed in language (`"a photo of S* with ... without ..."`).
2. **Calibrated hybrid** — the prompt is reduced to `"a photo of S*"` so `S*` carries identity only, and *both* the positive and negative constraints are applied in calibrated similarity space by adding/subtracting the z-scored attribute logits (`Z_ENS`, from the training-free part). This gives every query — including positive-only ones — a strong calibrated +/− signal that pure prompt composition cannot provide.


### Parameters

In [ ]:
P2W_SUBSET           = 30_000  # training images (subset of CelebA train split)
P2W_EPOCHS           = 10      # mapper training epochs
P2W_BATCH            = 128     # mini-batch size
P2W_LR               = 1e-4    # AdamW learning rate
P2W_MAPPER_HIDDEN    = 512     # hidden dim of the 3-layer MLP mapper
P2W_CONSTRAINT_WEIGHT = 0.5    # hybrid: weight of the calibrated +/- constraint term

### Training

#### Pic2Word model classes

In [ ]:
class Pic2WordTextEncoder(nn.Module):
    """Frozen-CLIP text encoder for prompts containing one pseudo-token.

    Same machinery as CoOpPromptLearner._encode_text_with_ctx — overwrite token
    embeddings, then run the frozen text transformer manually — but the injected
    vector is *per sample* (the mapper's output for one image) rather than a
    shared learned parameter.
    """

    PSEUDO_WORD = "x"

    def __init__(self, clip_model: CLIPModel, processor: CLIPProcessor):
        super().__init__()
        self.clip = clip_model
        self.tokenizer = processor.tokenizer
        self.max_len = clip_model.config.text_config.max_position_embeddings
        pseudo_ids = self.tokenizer(self.PSEUDO_WORD, add_special_tokens=False)["input_ids"]
        if len(pseudo_ids) != 1:
            raise ValueError(f"Pseudo word {self.PSEUDO_WORD!r} must map to a single token")
        self.pseudo_id = pseudo_ids[0]
        for p in self.clip.parameters():
            p.requires_grad_(False)

    def tokenize(self, prompts: list[str]) -> torch.Tensor:
        """Tokenize prompts, asserting each contains the pseudo word exactly once."""
        ids = self.tokenizer(
            prompts, padding="max_length", max_length=self.max_len,
            truncation=True, return_tensors="pt",
        )["input_ids"]
        if not bool(((ids == self.pseudo_id).sum(dim=1) == 1).all()):
            raise ValueError(f"Each prompt must contain {self.PSEUDO_WORD!r} exactly once: {prompts}")
        return ids

    def forward(self, input_ids: torch.Tensor, pseudo_emb: torch.Tensor) -> torch.Tensor:
        """input_ids: (B, L), or (1, L) broadcast over pseudo_emb (B, D_token).
        Returns (B, D) L2-normalised text features."""
        device_ = pseudo_emb.device
        input_ids = input_ids.to(device_)
        if input_ids.shape[0] == 1 and pseudo_emb.shape[0] > 1:
            input_ids = input_ids.expand(pseudo_emb.shape[0], -1)

        # Token embeddings (frozen) with the pseudo-token position overwritten
        # by the per-sample mapped vector.
        token_emb = self.clip.text_model.embeddings.token_embedding(input_ids).clone()
        rows, cols = (input_ids == self.pseudo_id).nonzero(as_tuple=True)
        token_emb[rows, cols] = pseudo_emb.to(token_emb.dtype)

        return _clip_encode_token_emb(self.clip, token_emb, input_ids)


class Pic2WordMapper(nn.Module):
    """3-layer MLP mapping a CLIP image embedding to a pseudo-token embedding."""

    def __init__(self, in_dim: int, out_dim: int, hidden: int = 512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, image_emb: torch.Tensor) -> torch.Tensor:
        return self.net(image_emb)


model, processor = get_CLIP_model()
p2w_encoder = Pic2WordTextEncoder(model, processor).to(device)
token_dim = model.text_model.embeddings.token_embedding.embedding_dim
mapper = Pic2WordMapper(in_dim=embeddings.shape[1], out_dim=token_dim, hidden=P2W_MAPPER_HIDDEN).to(device)
print(f"Mapper trainable parameters: {sum(p.numel() for p in mapper.parameters()):,}")


#### Pic2Word mapper training

In [ ]:
logit_scale_value = model.logit_scale.exp().detach()
optimizer = torch.optim.AdamW(mapper.parameters(), lr=P2W_LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=P2W_EPOCHS)

template_ids = p2w_encoder.tokenize(["a photo of x"])    # (1, 77), broadcast per batch

torch.manual_seed(0)
subset_idx = torch.randperm(train_features.shape[0], device=device)[:P2W_SUBSET]
X_p2w = train_features[subset_idx]                       # (S, 512), L2-normalized, frozen

print("Starting Pic2Word mapper training (a text-encoder forward per batch)...")
for epoch in range(P2W_EPOCHS):
    mapper.train()
    perm = torch.randperm(X_p2w.shape[0], device=device)
    total_loss = 0.0
    for start in range(0, X_p2w.shape[0], P2W_BATCH):
        x = X_p2w[perm[start:start + P2W_BATCH]]
        pseudo = mapper(x)                                # (B, D_token)
        t = p2w_encoder(template_ids, pseudo)             # (B, D)
        logits = logit_scale_value * (t @ x.T)
        labels_ce = torch.arange(x.shape[0], device=device)
        loss = 0.5 * (F.cross_entropy(logits, labels_ce) + F.cross_entropy(logits.T, labels_ce))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.detach()) * x.shape[0]
    scheduler.step()
    print(f"Epoch {epoch+1:3d}/{P2W_EPOCHS}  loss={total_loss / X_p2w.shape[0]:.4f}")

# Sanity check: encoding "a photo of S*" with S* mapped from an image should
# retrieve that very image from the gallery. Every probe in the training subset
# is checked; we only chunk to bound the (B x S) similarity matrix in memory.
mapper.eval()
with torch.no_grad():
    n_probe = X_p2w.shape[0]
    hits = 0
    for start in range(0, n_probe, 1024):   # chunk over probes; memory only
        sl = slice(start, min(start + 1024, n_probe))
        t = p2w_encoder(template_ids, mapper(X_p2w[sl]))
        top1 = (t @ X_p2w.T).argmax(dim=1)
        hits += int((top1 == torch.arange(start, sl.stop, device=device)).sum())
    self_rate = hits / n_probe
print(f"Self-retrieval top-1 rate over {n_probe} probes: {self_rate:.2%}")

P2W_CKPT = Path(EVALUATION_CACHE_DIR) / "pic2word_mapper.pt"
torch.save(
    {"state_dict": {k: v.cpu() for k, v in mapper.state_dict().items()}, "self_retrieval_top1": self_rate},
    P2W_CKPT,
)
print(f"Saved mapper to {P2W_CKPT}")


### Scorer

In [ ]:
def _humanize_attr(j: int) -> str:
    return idx2attribute[j].replace("_", " ").lower()


def build_pic2word_prompt(
    pos_idx: list[int],
    neg_idx: list[int],
    include_positives: bool = True,
    include_negatives: bool = True,
) -> str:
    """Compose the retrieval prompt around the pseudo-token, e.g.
    'a photo of x with eyeglasses without smiling'."""
    parts = ["a photo of x"]
    if include_positives and pos_idx:
        parts.append("with " + " and ".join(_humanize_attr(j) for j in pos_idx))
    if include_negatives and neg_idx:
        parts.append("without " + " or ".join(_humanize_attr(j) for j in neg_idx))
    return " ".join(parts)


def pic2word_scorer(
    embeddings: torch.Tensor,
    mapper: nn.Module,
    p2w_encoder: nn.Module,
    constraint_weight: float = 0.0,
    Z: torch.Tensor | None = None,
) -> Callable:
    """Scorer factory for Pic2Word.

    constraint_weight = 0  -> pure prompt-based retrieval: the +/- attributes are
        expressed entirely in language ('a photo of x with ... without ...').
    constraint_weight > 0  -> symmetric calibrated hybrid: the prompt is reduced to
        'a photo of x' so the pseudo-token S* carries identity only, and BOTH the
        positive and negative constraints are applied in calibrated similarity space
        via the z-scored attribute logits Z_ENS:
            score(x) = zscore(e_x . t_S*) + w*sum_{j in q+} Z[x,j] - w*sum_{j in q-} Z[x,j]
        This mirrors how the training-free methods handle constraints and gives every
        query (including positive-only ones) a calibrated +/- signal.

    Args:
        embeddings:        (N, D) L2-normalised gallery embeddings.
        mapper:            trained Pic2WordMapper.
        p2w_encoder:       Pic2WordTextEncoder (frozen CLIP).
        constraint_weight: weight of the calibrated +/- constraint term (hybrid only).
        Z:                 (N, n_attrs) z-scored attribute logits (required for hybrid).
    Returns:
        make_scorer(annotation) -> scorer(source_idx) -> (N,) score tensor.
    """
    mapper.eval()
    hybrid = constraint_weight > 0

    def make_scorer(annotation: dict) -> Callable:
        pos_idx, neg_idx = parse_query_signs(get_text_query(annotation))
        # Hybrid: identity-only prompt; constraints handled in similarity space.
        prompt = build_pic2word_prompt(
            pos_idx, neg_idx,
            include_positives=not hybrid,
            include_negatives=not hybrid,
        )
        ids    = p2w_encoder.tokenize([prompt])

        @torch.no_grad()
        def scorer(source_idx: int) -> torch.Tensor:
            pseudo = mapper(embeddings[source_idx].unsqueeze(0))
            t      = p2w_encoder(ids, pseudo).squeeze(0)
            scores = embeddings @ t
            scores = (scores - scores.mean()) / scores.std().clamp_min(1e-6)
            if hybrid:
                if pos_idx:
                    scores = scores + constraint_weight * Z[:, pos_idx].sum(dim=1)
                if neg_idx:
                    scores = scores - constraint_weight * Z[:, neg_idx].sum(dim=1)
            return scores

        return scorer
    return make_scorer

### Evaluation & Plot

In [ ]:
evaluation_results_p2w = evaluate(
    annotations,
    pic2word_scorer(embeddings, mapper, p2w_encoder),
)
average_results_per_query_p2w = [
    compute_query_average_results(q) for q in evaluation_results_p2w
]

evaluation_results_p2w_hybrid = evaluate(
    annotations,
    pic2word_scorer(embeddings, mapper, p2w_encoder, constraint_weight=P2W_CONSTRAINT_WEIGHT, Z=Z_ENS),
)
average_results_per_query_p2w_hybrid = [
    compute_query_average_results(q) for q in evaluation_results_p2w_hybrid
]

plot_methods_comparison(
    {
        "Pic2Word (prompt only)": average_results_per_query_p2w,
        "Pic2Word (calibrated +/-)": average_results_per_query_p2w_hybrid,
    },
    title="Pic2Word — prompt-only vs. calibrated +/- hybrid",
)


## Final Comparison — all methods

All methods evaluated on the same benchmark JSON and the same precomputed image embeddings: the training-free series (baseline, source-profile matching, prompt ensembling) followed by the training-based methods (CoOp, Conditional Combiner, Pic2Word).


In [ ]:
plot_methods_comparison(
    {
        "Baseline":                            average_results_per_query_baseline,
        "Source-Profile Matching":                average_results_per_query_profmatch,
        "Prompt Ensembling":  average_results_per_query_promptens,
        f"CoOp (M={COOP_M})":                  average_results_per_query_coop,
        "Combiner":                            average_results_per_query_combiner,
        "Pic2Word (hybrid)":                   average_results_per_query_p2w_hybrid,
    },
    title="Final Method Comparison — per-query Recall@K and Precision@K",
)